# Use Case — Public-Parks Heat-Resilience Audit

**Who this is for**  
City parks-and-recreation directors, public-health environmental health teams, and climate-resilience officers. The persona is the same person who would commission a tree-canopy survey or a playground shade audit — but here the input is a **parks GIS export** and the output is a per-park diagnosis tied to nationally-funded improvement programs.

**The scenario**  
Summer is here. Park usage drops on hot days, which has measurable public-health consequences: less physical activity (a CDC-tracked outcome), and for the people who *do* go, real heat-illness risk. Parks in neighborhoods where homes lack AC are *de facto* cooling refuges — when those parks are hot, residents have nowhere to go. You manage a list of parks; you need to know which ones are exposing the public to dangerous heat, **why** each is hot, and **what** specific improvement to fund — without inventing a score and without quoting a dollar figure that doesn't generalize from one city to another.

This notebook combines **your park list** with **FortyGuard layers** to answer four public-health questions:

1. **Where is it hot, when, and by how much?**  ← 24-hour heatmap × your park points
2. **Why is each park hot?**  ← satellite segmentation on the top exposures (canopy %, impervious %)
3. **What does it look like at ground level where the public actually stands?**  ← street view at the worst park
4. **Is the heat a humidity problem (heat-index) or a dry-heat problem during use hours?**  ← env-params at the top exposures

**The output is declarative, not synthetic.** Every column you see is a direct API measurement (temperature, heat index, canopy %, impervious %, sky %). Every recommendation is a *threshold trigger* — *if measurement X crosses a published NOAA / EPA / CDC / USDA / NRPA threshold, then recommend the program that funds the fix*. No invented index. No dollar value. Portable to any city in the country.

> **Runs live against the API.** Add your `FORTYGUARD_API_KEY` to `.env` and place your parks CSV at `data/sample_public_parks.csv`. The `data/` directory is git-ignored and **not** shipped with the repo — bring your own input (see the schema below).

> **Bring your own data.** Put a parks CSV at `data/sample_public_parks.csv` (the `data/` directory is git-ignored — not shipped). As long as the columns match (`park_id`, `name`, `type`, `acres`, `latitude`, `longitude`), everything downstream works; swap the path in Step 1 to use your own.

> **U.S. coverage only.** All FortyGuard endpoints operate over locations inside the United States. Swap the AOI to any U.S. city — coordinates outside the U.S. will return errors or empty responses.

> **Dates: 2021 to today.** `STUDY_DATE` must be on or after `2021-01-01` (the catalog's start) and no later than today; earlier or future dates fail at the heatmap call with a "no data available" error.

**Why this is different from the existing facility-cooling notebook**: that one weights facility points by *vulnerable_population* (a number you have to supply per facility). This one needs only what every parks office already publishes — a list of parks with coordinates — and explains heat in terms of the *physical environment* the API directly measures.

---

## Setup

In [ ]:
import sys, pathlib, time as _time
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import json
import numpy as np
import pandas as pd
import folium
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import Point, shape, mapping
from IPython.display import HTML, display

from fortyguard import FortyGuardClient
from fortyguard.exceptions import FortyGuardError

# ── configuration ─────────────────────────────────────────
STUDY_DATE        = '2024-10-02'         # change to the day you want to prioritize on
STUDY_HOUR        = '14:00'              # design-peak afternoon — required by heatmap & satellite calls
GRANULARITY_M     = 80                   # heatmap resolution
TOP_N_TO_ENRICH   = 3                    # depth-of-diagnosis budget (satellite + street + env-params)
USE_HOUR_START    = 10                   # daytime use window for heat-index thresholding
USE_HOUR_END      = 18                   # 18:00 inclusive (covers afternoon + early-evening peak)
TEMP_C_SANITY     = (15.0, 55.0)         # F→C sanity range for the cached heatmap

# ── published thresholds (no invented numbers — every value below comes from a public source) ─
# Heat Index categories: NOAA NWS Heat Index Chart
HI_CAUTION_C          = 27.0   # NOAA 'Caution' boundary (heat index ≥ 27 °C / 80 °F)
HI_EXTREME_CAUTION_C  = 32.0   # NOAA 'Extreme Caution' boundary (heat index ≥ 32 °C / 90 °F)
HI_DANGER_C           = 39.0   # NOAA 'Danger' boundary (heat index ≥ 39 °C / 103 °F)
# Surface composition: EPA Heat Island Reduction guidance + USDA i-Tree program targets
CANOPY_TARGET_PCT     = 25.0   # EPA / USDA target tree canopy in urban park settings
IMPERVIOUS_TRIGGER    = 60.0   # EPA Heat Island guidance: > 60% impervious is a cool-pavement candidate
# Ground-level: NRPA shade-equity guidance
SKY_TRIGGER_PCT       = 60.0   # NRPA: sky fraction > 60% in front-view of play area = unshaded
# Activity guidance: CDC physical-activity + heat guidance
RH_HIGH_PCT           = 60.0   # > 60% RH at peak combined with > 30 °C ambient triggers splash-pad / mister review

# ── published-program citation strings (used in the action brief, no values invented) ────────
PROG_USDA_ITREE       = 'USDA Forest Service i-Tree · EPA Heat Island Reduction (Trees & Vegetation)'
PROG_EPA_HEATISLAND   = 'EPA Heat Island Reduction (Cool Pavements)'
PROG_NRPA_SHADE       = 'NRPA Shade-Equity guidance · CDC BRACE shade-structure programs'
PROG_CDC_BRACE        = 'CDC BRACE — heat advisory + activity-window guidance'
PROG_CDC_HEAT_RES     = 'CDC heat-resilience programs (splash pads / misters)'

# ── data paths — outputs and caches are organized by endpoint family ───
DATA            = ROOT / 'data'
PARKS_CSV       = DATA / 'sample_public_parks.csv'

HEATMAP_DIR     = DATA / 'heatmaps'
SAT_SEG_DIR     = DATA / 'satellite'
STREET_SEG_DIR  = DATA / 'street_view'
ENV_DIR         = DATA / 'env_params'
for d in (HEATMAP_DIR, SAT_SEG_DIR, STREET_SEG_DIR, ENV_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Default cache file for each endpoint — change the filename to load any captured run.
HEATMAP_GEOJSON = HEATMAP_DIR    / 'real_estate_san_jose_heatmap_sample_day_2024-10-02.geojson'
SATELLITE_JSON  = SAT_SEG_DIR    / 'real_estate_san_jose_satellite_segmentation_sample_day_2024-10-02.json'
STREETVIEW_JSON = STREET_SEG_DIR / 'real_estate_san_jose_street_view_segmentation_sample_day_2024-10-02.json'
ENV_PARAMS_JSON = ENV_DIR        / 'real_estate_san_jose_env_paramaters_sample_day_2024-10-02.json'

# Live client — only needed for Step 2a / 6a / 7a / 8a (the cached cells run offline).
try:
    client = FortyGuardClient()
    HAVE_API = True
except FortyGuardError as exc:
    client = None
    HAVE_API = False
    print(f'[no API key] live cells will not work: {exc}')


# --- Quiet polling helper ----------------------------------------------------
# Each live API call prints exactly two lines: "Submitted ..." and "✓ ... completed."
# The API occasionally returns 403 "Unauthorized access" on the first poll before the
# activity is registered — we sleep briefly and retry up to a few times before giving up.
def submit_and_wait_quiet(client_method, label, *, poll_interval=8.0,
                          initial_delay=3.0, transient_403_retries=4,
                          failure_retries=2, timeout=600.0,
                          skip_on_failure=False, **kwargs):
    # Some analysis tasks (esp. satellite / street-view) intermittently fail or
    # hang on the backend, and some points have no imagery at all. Retry the whole
    # submit up to failure_retries times on a task-level failure; with
    # skip_on_failure=True, return None once retries/timeout are exhausted so a
    # caller loop can skip that point instead of aborting the whole step.
    def _giveup(exc):
        if skip_on_failure:
            print(f"  ⤼ {label} unavailable — skipping ({type(exc).__name__}).")
            return None
        print(f"  ✗ {label} failed: {exc}")
        raise exc
    for submit_attempt in range(failure_retries + 1):
        activity_id = client_method(wait=False, **kwargs)
        print(f"Submitted {label} → {activity_id}")

        if initial_delay:
            _time.sleep(initial_delay)

        for attempt in range(transient_403_retries + 1):
            try:
                result = client.wait_for(activity_id, poll_interval=poll_interval, timeout=timeout)
                print(f"  ✓ {label} completed.")
                return {"activity_id": activity_id, "result": result}
            except FortyGuardError as exc:
                msg = str(exc)
                is_403 = "-> 403" in msg or "Unauthorized access" in msg
                is_task_failure = " failed:" in msg and "-> " not in msg
                if is_403 and attempt < transient_403_retries:
                    back_off = 5 * (attempt + 1)
                    print(f"  ⏳ status returned 403 (transient); retrying in {back_off}s…")
                    _time.sleep(back_off)
                    continue
                if is_task_failure and submit_attempt < failure_retries:
                    back_off = 5 * (submit_attempt + 1)
                    print(f"  ↻ task failed (transient backend error); re-submitting in {back_off}s…")
                    _time.sleep(back_off)
                    break  # re-submit via the outer loop
                return _giveup(exc)
            except Exception as exc:
                return _giveup(exc)
    return _giveup(RuntimeError(f"{label}: exhausted all submit attempts"))


# ── shared color ramp ────────────────────────────────────────
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
TCM_CMAP = LinearSegmentedColormap.from_list('tcm', TCM_COLORS, N=256)
def temp_color(t, lo, hi):
    if t is None or hi == lo: return TCM_COLORS[0]
    frac = max(0.0, min(1.0, (float(t) - lo) / (hi - lo)))
    return TCM_COLORS[min(int(frac * len(TCM_COLORS)), len(TCM_COLORS) - 1)]

PARK_TYPE_PALETTE = {
    'Regional'    : '#1f77b4',
    'Neighborhood': '#2ca02c',
    'Plaza'       : '#9467bd',
    'Sports'      : '#ff7f0e',
    'Playground'  : '#e377c2',
    'Trail-head'  : '#8c564b',
    'Dog-Park'    : '#7f7f7f',
}

# Output bundle root for Step 11 — populated as cells run.
OUTPUTS_ROOT = ROOT / 'outputs' / f'parks_{STUDY_DATE}'
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV   = OUTPUTS_ROOT / 'audit.csv'

print(f'STUDY_DATE={STUDY_DATE}  STUDY_HOUR={STUDY_HOUR}  TOP_N_TO_ENRICH={TOP_N_TO_ENRICH}')
print(f'Use-hour window for heat-index thresholding: {USE_HOUR_START:02d}:00 – {USE_HOUR_END:02d}:00')
print(f'Output bundle root: {OUTPUTS_ROOT.relative_to(ROOT)}')


---
## Step 1 — Load your park list

### What you are doing
Reading a parks point CSV. The schema is the minimum that any city's parks GIS already exports — `park_id`, `name`, `type`, `acres`, `latitude`, `longitude`. No demographic columns required, no vulnerability weights to author by hand.

### Why this matters (public health)
Starting from your own park list keeps the audit portable. Parks departments in different states maintain different fields; the only ones the heat analysis actually needs are coordinates. Everything else passes through to the final CSV so the recommendations land in *your* GIS, not in a stand-alone deliverable that has to be hand-merged later.

In [ ]:
parks = pd.read_csv(PARKS_CSV)
type_counts = parks['type'].value_counts().to_dict()
print(f"Loaded {len(parks)} parks covering {parks['acres'].sum():.0f} total acres")
print(f"  type mix: {type_counts}")
parks

---
## Step 2a — Heat layer (via live API)

### What you are doing
Calling `client.create_heatmap` with `filter_type=3` (single day — covers the full 24 h; `start_time` is ignored). The response is a GeoJSON tile layer where every tile carries a daily aggregate (`min_temperature`, `max_temperature`, `average_temperature`) in **°C**. We persist the raw response under `data/heatmaps/` for reuse and use the daily MAX (peak) as each tile's `temperature` so downstream uses a single field. (The Enterprise API returns aggregates only — no hourly `'00'..'23'` fields.)

### Why this matters
Daily peak — not daily average — is what heat exposure and the action brief should rank against. Average temperature drags in cool nighttime hours and underweights afternoon heat; peak is what visitors actually feel during the use window.

In [ ]:
import textwrap

def _as_c(f):
    return None if f is None else float(f)


def show_heatmap_summary(temps, source_label):
    """Stats card + colored histogram + vertical colorbar — same visual as the
    real-estate and bus-stops notebooks."""
    temps = [t for t in temps if t is not None]
    if not temps:
        print('No tile temperatures to summarize.')
        return
    lo, hi = float(min(temps)), float(max(temps))
    mean   = float(sum(temps) / len(temps))

    wrapped_label = textwrap.fill(source_label, width=22) if source_label else ''
    n_label_lines = wrapped_label.count('\n') + 1

    fig = plt.figure(figsize=(12, 3.4 + 0.30 * max(0, n_label_lines - 1)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    ax0.text(0.0, 0.97, wrapped_label, transform=ax0.transAxes,
             fontsize=10.5, fontweight='bold', color='#222', va='top')
    subtitle_y = 0.97 - 0.11 * n_label_lines - 0.05
    ax0.text(0.0, subtitle_y, f'{len(temps):,} tiles',
             transform=ax0.transAxes,
             fontsize=10, color='#666', va='top')

    rows = [('min',  lo,   temp_color(lo,   lo, hi)),
            ('mean', mean, temp_color(mean, lo, hi)),
            ('max',  hi,   temp_color(hi,   lo, hi))]
    band_top    = subtitle_y - 0.10
    band_bottom = 0.05
    step        = (band_top - band_bottom) / max(len(rows) - 1, 1)
    rect_h      = min(0.14, step * 0.6)
    y = band_top
    for label, val, color in rows:
        ax0.text(0.0, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace')
        ax0.add_patch(plt.Rectangle((0.22, y - rect_h / 2), 0.10, rect_h,
                                    transform=ax0.transAxes,
                                    facecolor=color, edgecolor='#333', linewidth=0.6))
        ax0.text(0.37, y, f'{val:.2f} °C', transform=ax0.transAxes,
                 fontsize=13, fontweight='bold', color='#222', va='center',
                 family='monospace')
        y -= step

    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, edge_lo, edge_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((edge_lo + edge_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for spine in ('top', 'right'):
        ax1.spines[spine].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP,
               extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([])
    ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)

    plt.show()


# --- AOI bounding box around the parks (small buffer) ---------------------
rx0 = parks['longitude'].min() - 0.005
rx1 = parks['longitude'].max() + 0.005
ry0 = parks['latitude' ].min() - 0.005
ry1 = parks['latitude' ].max() + 0.005
aoi = {'type': 'FeatureCollection', 'features': [{
    'type': 'Feature', 'properties': {},
    'geometry': {'type': 'Polygon', 'coordinates': [[
        [rx0, ry0], [rx1, ry0], [rx1, ry1], [rx0, ry1], [rx0, ry0]]]},
}]}

if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY in .env. Run Step 2b instead.')

heatmap = submit_and_wait_quiet(
    client.create_heatmap,
    f'heatmap {STUDY_DATE}',
    polygon_aoi=aoi,
    start_date=STUDY_DATE,
    start_time=STUDY_HOUR,
    filter_type=3,                  # single day — daily aggregate per tile
    granularity=GRANULARITY_M,
)

map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []

# --- Persist the raw response under data/heatmaps/. ---
LIVE_HEATMAP_PATH = HEATMAP_DIR / f'heatmap_san_jose_{STUDY_DATE}_live.geojson'
with open(LIVE_HEATMAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(map_data, f)
print(f'Saved raw heatmap → {LIVE_HEATMAP_PATH.relative_to(ROOT)}')

# --- Normalize to (poly, hourly_c, peak_c, peak_h, mean_c) tuples. ---
# filter_type=3 returns per-tile daily aggregates (min/max/average_temperature) in °C.
# The Enterprise API does not emit hourly '00'..'23' fields, so hourly_c falls back
# to the daily peak and peak_hour is None — hours_above_caution_in_use / diurnal
# signals degrade to the daily aggregate (they'd need hourly data this API omits).
tiles, minx, miny, maxx, maxy = [], 1e9, 1e9, -1e9, -1e9
for ft in features:
    poly  = shape(ft['geometry'])
    props = ft.get('properties', {}) or {}
    if all(f'{h:02d}' in props for h in range(24)):
        hourly_c = [_as_c(props[f'{h:02d}']) for h in range(24)]
        peak_c   = max(v for v in hourly_c if v is not None)
        peak_h   = next((i for i, v in enumerate(hourly_c) if v == peak_c), None)
        mean_c   = sum(hourly_c) / len(hourly_c)
    elif 'max_temperature' in props:
        peak_c   = _as_c(props.get('max_temperature'))
        if peak_c is None:
            continue
        mean_c   = _as_c(props.get('average_temperature')) or peak_c
        hourly_c = [peak_c] * 24
        peak_h   = None
    else:
        continue
    tiles.append((poly, hourly_c, round(peak_c, 2), peak_h, round(mean_c, 2)))
    x0, y0, x1, y1 = poly.bounds
    minx, miny = min(minx, x0), min(miny, y0)
    maxx, maxy = max(maxx, x1), max(maxy, y1)

aoi_bounds = (minx, miny, maxx, maxy)
peaks = [t[2] for t in tiles]
has_diurnal = any(t[3] is not None for t in tiles)
print(f'[live] {len(tiles):,} tiles, peak range {min(peaks):.1f}..{max(peaks):.1f} °C')
print(f'[live] hourly tile data: {"present" if has_diurnal else "absent"} '
      f'(peak_hour and hours-above-caution will be {"populated" if has_diurnal else "None"})')
show_heatmap_summary(peaks, f'Live API · {STUDY_DATE} (daily peak)')


def tile_for(lat, lon):
    p = Point(lon, lat)
    for t in tiles:
        if t[0].contains(p): return t
    return min(tiles, key=lambda t: t[0].centroid.distance(p))


---
## Step 2b — Or: load a cached heatmap (for testing)

### What you are doing
Loading a captured heatmap GeoJSON from `data/heatmaps/`. Each tile carries daily aggregates (`min`/`max`/`average_temperature`) in **°C**. We use `max_temperature` as each tile's peak `temperature`. The Enterprise API returns aggregates only — no hourly `'00'..'23'` — so diurnal-derived signals (peak_hour, hours-above-caution) degrade to the daily peak.

### Why this matters
Use this path when iterating on the analysis without burning API credits, or when working with an older capture that carries hourly tile data the current API mode doesn't expose.

In [ ]:
import textwrap

def _as_c(f):
    return None if f is None else float(f)


def show_heatmap_summary(temps, source_label):
    """Stats card + colored histogram + vertical colorbar — same visual as 2a."""
    temps = [t for t in temps if t is not None]
    if not temps:
        print('No tile temperatures to summarize.')
        return
    lo, hi = float(min(temps)), float(max(temps))
    mean   = float(sum(temps) / len(temps))

    wrapped_label = textwrap.fill(source_label, width=22) if source_label else ''
    n_label_lines = wrapped_label.count('\n') + 1

    fig = plt.figure(figsize=(12, 3.4 + 0.30 * max(0, n_label_lines - 1)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    ax0.text(0.0, 0.97, wrapped_label, transform=ax0.transAxes,
             fontsize=10.5, fontweight='bold', color='#222', va='top')
    subtitle_y = 0.97 - 0.11 * n_label_lines - 0.05
    ax0.text(0.0, subtitle_y, f'{len(temps):,} tiles',
             transform=ax0.transAxes,
             fontsize=10, color='#666', va='top')

    rows = [('min',  lo,   temp_color(lo,   lo, hi)),
            ('mean', mean, temp_color(mean, lo, hi)),
            ('max',  hi,   temp_color(hi,   lo, hi))]
    band_top    = subtitle_y - 0.10
    band_bottom = 0.05
    step        = (band_top - band_bottom) / max(len(rows) - 1, 1)
    rect_h      = min(0.14, step * 0.6)
    y = band_top
    for label, val, color in rows:
        ax0.text(0.0, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace')
        ax0.add_patch(plt.Rectangle((0.22, y - rect_h / 2), 0.10, rect_h,
                                    transform=ax0.transAxes,
                                    facecolor=color, edgecolor='#333', linewidth=0.6))
        ax0.text(0.37, y, f'{val:.2f} °C', transform=ax0.transAxes,
                 fontsize=13, fontweight='bold', color='#222', va='center',
                 family='monospace')
        y -= step

    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, edge_lo, edge_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((edge_lo + edge_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for spine in ('top', 'right'):
        ax1.spines[spine].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP,
               extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([])
    ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)

    plt.show()


with open(HEATMAP_GEOJSON, 'r', encoding='utf-8') as f:
    map_data = json.load(f)

# Enterprise API returns per-tile daily aggregates (min/max/average_temperature) in °C — no hourly '00'..'23'. Build (poly, hourly_c, peak, peak_h, mean) tuples.
tiles, minx, miny, maxx, maxy = [], 1e9, 1e9, -1e9, -1e9
for ft in map_data.get('features', []):
    poly  = shape(ft['geometry'])
    props = ft.get('properties', {}) or {}
    if all(f'{h:02d}' in props for h in range(24)):
        hourly_c = [_as_c(props[f'{h:02d}']) for h in range(24)]
    elif 'max_temperature' in props:
        peak_c   = _as_c(props['max_temperature'])
        hourly_c = [peak_c] * 24
    else:
        continue
    peak_c = max(t for t in hourly_c if t is not None)
    peak_h = next((i for i, v in enumerate(hourly_c) if v == peak_c), None)
    mean_c = sum(hourly_c) / len(hourly_c)
    tiles.append((poly, hourly_c, round(peak_c, 2), peak_h, round(mean_c, 2)))
    x0, y0, x1, y1 = poly.bounds
    minx, miny = min(minx, x0), min(miny, y0)
    maxx, maxy = max(maxx, x1), max(maxy, y1)

aoi_bounds = (minx, miny, maxx, maxy)
peaks = [t[2] for t in tiles]
lo, hi = min(peaks), max(peaks)
assert TEMP_C_SANITY[0] <= lo and hi <= TEMP_C_SANITY[1], \
    f'F→C sanity check failed: peak range {lo:.1f}..{hi:.1f} outside {TEMP_C_SANITY}'
print(f'[cached] {len(tiles):,} tiles, peak range {lo:.1f}..{hi:.1f} °C')
print(f'[cached] AOI bounds (lon,lat): ({aoi_bounds[0]:.4f}, {aoi_bounds[1]:.4f}) → '
      f'({aoi_bounds[2]:.4f}, {aoi_bounds[3]:.4f})')
show_heatmap_summary(peaks, f'Cached · {HEATMAP_GEOJSON.name}')


def tile_for(lat, lon):
    p = Point(lon, lat)
    for t in tiles:
        if t[0].contains(p): return t
    return min(tiles, key=lambda t: t[0].centroid.distance(p))


---
## Step 3 — Diurnal temperature attach

### What you are doing
For each park, find the heatmap tile that contains its coordinates and copy off **peak temperature**, **peak hour**, **daily-mean temperature**, **diurnal swing**, **AOI percentile**, and **hours-above-Caution-during-use-window** — the count of hours in 10:00–18:00 when the tile reads ≥ 27 °C (NOAA Heat-Index Caution boundary). Now every park has an analysis-ready row driven entirely by API output.

### Why this matters (public health)
*Hours-above-Caution-during-use-window* is the column that connects the heat measurement to a public-health consequence. NOAA Caution means "fatigue possible with prolonged exposure" — every hour past that boundary is an hour during which a parent has to ask whether the playground visit is safe. A peak of 35 °C at 03:00 doesn't matter; a peak of 33 °C from 13:00 to 18:00 means the park is functionally unusable for half its operating day.

In [ ]:
def _percentile_rank(value, sorted_values):
    lo, hi = 0, len(sorted_values)
    while lo < hi:
        mid = (lo + hi) // 2
        if sorted_values[mid] <= value: lo = mid + 1
        else: hi = mid
    return round(100.0 * lo / max(1, len(sorted_values)), 1)

aoi_peak_sorted = sorted(t[2] for t in tiles)
use_hours = list(range(USE_HOUR_START, USE_HOUR_END + 1))


def _hourly_derivable(hourly_c):
    """Return True if hourly_c carries real per-hour variation (cache path)."""
    if not hourly_c or len(hourly_c) < 24:
        return False
    return min(hourly_c) != max(hourly_c)


records = []
for _, r in parks.iterrows():
    poly, hourly_c, peak_c, peak_h, mean_c = tile_for(r.latitude, r.longitude)
    has_diurnal = _hourly_derivable(hourly_c)
    if has_diurnal:
        use_window_temps  = [hourly_c[h] for h in use_hours]
        hours_above_caut  = sum(1 for v in use_window_temps if v is not None and v >= HI_CAUTION_C)
        use_window_peak_c = round(max(use_window_temps), 1)
        diurnal_swing_c   = round(peak_c - min(hourly_c), 1)
    else:
        # Live filter_type=3: no hourly tile temps. Step 8 still provides hourly heat-index from env-params.
        hours_above_caut  = None
        use_window_peak_c = None
        diurnal_swing_c   = None
    records.append({
        'peak_temp_c'                  : round(peak_c, 1),
        'peak_hour'                    : peak_h,
        'daily_mean_temp_c'            : round(mean_c, 1),
        'diurnal_swing_c'              : diurnal_swing_c,
        'aoi_percentile'               : _percentile_rank(peak_c, aoi_peak_sorted),
        'hours_above_caution_in_use'   : hours_above_caut,
        'use_window_peak_c'            : use_window_peak_c,
    })
parks = pd.concat([parks.reset_index(drop=True), pd.DataFrame(records)], axis=1)
parks = parks.sort_values('peak_temp_c', ascending=False).reset_index(drop=True)
parks.insert(0, 'rank', parks.index + 1)

parks[['rank', 'park_id', 'name', 'type', 'acres',
       'peak_temp_c', 'peak_hour', 'daily_mean_temp_c',
       'aoi_percentile', 'hours_above_caution_in_use', 'use_window_peak_c']]


---
## Step 4 — Park-network overview map (M1)

### What you are doing
Drop the AOI heatmap (daily-average temperature, equal-interval classes) under your park markers. Marker size scales with peak temperature; color encodes park type so the parks director can see *which kinds of public spaces* are sitting in the heat — a regional park with high attendance vs. a small playground gets different attention.

### Why this matters (public health)
This is the briefing slide. Before any per-park diagnosis, the director sees the network in city context — the same way an incident commander sees an operational map. Type-color encoding catches systemic patterns: "all the playgrounds happen to be in the hot quarter of town" is a finding even before you look at any single park.

In [ ]:
N_BINS = len(TCM_COLORS)
tile_avgs = [t[4] for t in tiles]
least, highest = min(tile_avgs), max(tile_avgs)
interval = (highest - least) / N_BINS if highest > least else 0.0
class_entries = [
    {'min': least + i * interval,
     'max': least + (i + 1) * interval,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]
def _class_color(t):
    if t is None or interval == 0: return class_entries[0]['color']
    idx = int((t - least) / interval)
    return class_entries[max(0, min(idx, N_BINS - 1))]['color']

heatmap_features = [{
    'type': 'Feature',
    'geometry': mapping(poly),
    'properties': {
        'avg_str': f'{avg:.2f} °C',
        'fillColor': _class_color(avg),
    },
} for (poly, _, _, _, _), avg in zip(tiles, tile_avgs)]


def _peak_hour_str(p):
    return f'@ {int(p.peak_hour):02d}:00' if pd.notna(p.peak_hour) else '(daily peak)'


def _hours_above_str(p):
    v = p.get('hours_above_caution_in_use') if hasattr(p, 'get') else getattr(p, 'hours_above_caution_in_use', None)
    return f'{int(v)}' if pd.notna(v) else '—'


min_peak = parks['peak_temp_c'].min()
center = [parks['latitude'].mean(), parks['longitude'].mean()]
m1 = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': heatmap_features},
    style_function=lambda f: {
        'fillColor': f['properties']['fillColor'],
        'color':     f['properties']['fillColor'],
        'weight':    0.6, 'fillOpacity': 0.75, 'opacity': 1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Avg temp'], sticky=True),
).add_to(m1)
for _, p in parks.iterrows():
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=4 + (p.peak_temp_c - min_peak) * 1.4,
        color='#000', weight=1.2,
        fill=True, fill_color=PARK_TYPE_PALETTE.get(p['type'], '#888'),
        fill_opacity=0.95,
        popup=(f"<b>#{int(p['rank'])} {p['name']}</b><br/>"
               f"{p['type']} · {p['acres']:.1f} acres<br/>"
               f"peak: {p.peak_temp_c:.1f}°C {_peak_hour_str(p)}<br/>"
               f"hours ≥ NOAA Caution (10–18): {_hours_above_str(p)}<br/>"
               f"AOI percentile: {p['aoi_percentile']}"),
    ).add_to(m1)

bbox_pad = 0.005
m1.fit_bounds([[parks['latitude'].min() - bbox_pad, parks['longitude'].min() - bbox_pad],
               [parks['latitude'].max() + bbox_pad, parks['longitude'].max() + bbox_pad]])

# --- type-color legend ---------------------------------------------------
legend_rows = ''.join(
    f'<tr>'
    f'<td style="padding:2px 6px;"><span style="display:inline-block;width:12px;height:12px;'
    f'background:{color};border:1px solid #000;"></span></td>'
    f'<td style="padding:2px 8px;">{label}</td></tr>'
    for label, color in PARK_TYPE_PALETTE.items()
)
m1.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;'
    'padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Park type</b><br/>'
    '<span style="color:#666;">marker color = park type · marker size ∝ daily peak temperature</span>'
    f'<table style="margin-top:6px;border-collapse:collapse;">{legend_rows}</table>'
    '</div>'
))
display(m1)


---
## Step 5 — Above-median hot exposures (M2)

### What you are doing
Filter to parks whose peak temperature is at or above the network median, and render only the heatmap tiles those parks sit on. This isolates "the parks we have to talk about" from the rest.

### Why this matters (public health)
A parks director cannot diagnose 12 parks at the depth of satellite + street view + env-params. Step 5 picks the ones where the heat measurement already justifies a closer look — *before* spending a single API call on satellite or env-params. The cut is empirical (the median splits the network in half), not invented.

In [ ]:
median_peak = parks['peak_temp_c'].median()
hot = parks[parks['peak_temp_c'] >= median_peak].copy()

joined, seen = [], set()
for _, p in hot.iterrows():
    pt = Point(p.longitude, p.latitude)
    matched = next((t for t in tiles if t[0].contains(pt)), None)
    if matched is None:
        continue
    poly, _, peak_c, peak_h, _ = matched
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen: continue
    seen.add(key)
    peak_str = (f'{peak_c:.2f} °C peak @ {peak_h:02d}:00'
                if peak_h is not None else f'{peak_c:.2f} °C daily peak')
    joined.append({'type':'Feature', 'geometry': mapping(poly),
                   'properties':{'peak_str': peak_str,
                                 'fillColor': _class_color(peak_c)}})

m2 = folium.Map(location=center, zoom_start=13, tiles='cartodbpositron')
folium.GeoJson(
    {'type':'FeatureCollection','features':joined},
    style_function=lambda f: {
        'fillColor': f['properties']['fillColor'],
        'color':     '#000', 'weight': 1.5, 'fillOpacity': 0.9,
    },
    tooltip=folium.GeoJsonTooltip(fields=['peak_str'], aliases=['Tile peak'], sticky=True),
).add_to(m2)
for _, p in hot.iterrows():
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=8, color='black', weight=1.2,
        fill=True, fill_color=PARK_TYPE_PALETTE.get(p['type'], '#888'), fill_opacity=0.95,
        popup=(f"<b>#{int(p['rank'])} {p['name']}</b><br/>"
               f"{p['type']} · {p['acres']:.1f} acres<br/>"
               f"peak: {p.peak_temp_c:.1f}°C {_peak_hour_str(p)}<br/>"
               f"hours ≥ NOAA Caution (10–18): {_hours_above_str(p)}"),
    ).add_to(m2)
if not hot.empty:
    m2.fit_bounds([[hot['latitude'].min() - 0.004, hot['longitude'].min() - 0.004],
                   [hot['latitude'].max() + 0.004, hot['longitude'].max() + 0.004]])

print(f'{len(hot)} parks at or above the network median peak ({median_peak:.1f}°C); '
      f'{len(joined)} unique heatmap tiles after the spatial join.')
display(m2)
hot[['rank','park_id','name','type','peak_temp_c','peak_hour','hours_above_caution_in_use','aoi_percentile']]


---
## Step 6a — Surface diagnosis (live, top-N)

### What you are doing
For each of the top-N hottest parks, calling `client.satellite_segmentation` with `filter_type=3` + `start_time=STUDY_HOUR`. We persist each raw response under `data/satellite/`, decode the original + segmented imagery, and bucket the class % into canopy / impervious / grass.

### Why this matters
Knowing a park is hot is not actionable on its own — *intervention selection depends on the cause*. Planting trees fixes a low-canopy problem; reflective paving fixes a high-impervious problem. Satellite segmentation tells you which driver dominates per park.

In [ ]:
import base64, io
from PIL import Image

IMPERV_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings',
               'rooftop', 'rooftops', 'sidewalk', 'earth', 'bare', 'ground'}
CANOPY_KEYS = {'tree', 'trees', 'vegetation', 'greenery', 'park'}
GRASS_KEYS  = {'grass'}


def _bucket(segments, keys):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)


def _first_b64(value):
    if isinstance(value, list): return value[0] if value else None
    return value


def _decode_b64(b64_str):
    if not b64_str: return None
    if isinstance(b64_str, list): b64_str = b64_str[0]
    if b64_str.startswith('data:'): b64_str = b64_str.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))


if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY. Run Step 6b instead.')

top_n = parks.head(TOP_N_TO_ENRICH).copy()
seg_data = {}
sat_imgs = {}
sat_source = {}

for _, r in top_n.iterrows():
    sat = submit_and_wait_quiet(
        client.satellite_segmentation,
        f"satellite #{int(r['rank'])} {r.park_id}",
        latitude=float(r.latitude), longitude=float(r.longitude),
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=3, granularity=GRANULARITY_M,
    )
    res = sat['result']

    # Persist the raw response under data/satellite/.
    out_path = SAT_SEG_DIR / f'satellite_{r.park_id}_{STUDY_DATE}_live.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f'  saved → {out_path.relative_to(ROOT)}')

    seg_block = res.get('segmentation', {}) or {}
    seg_data[r.park_id] = seg_block.get('segments', {}) or {}
    sat_imgs[r.park_id] = {'orig': _first_b64(res.get('orignal_image') or res.get('original_image')),
                           'seg' : seg_block.get('image_content'),
                           'legend': seg_block.get('image_legend', {}) or {}}
    sat_source[r.park_id] = 'live'

top_n['canopy_pct']     = top_n['park_id'].map(
    lambda pid: _bucket(seg_data.get(pid), CANOPY_KEYS) if pid in seg_data else None)
top_n['impervious_pct'] = top_n['park_id'].map(
    lambda pid: _bucket(seg_data.get(pid), IMPERV_KEYS) if pid in seg_data else None)
top_n['grass_pct']      = top_n['park_id'].map(
    lambda pid: _bucket(seg_data.get(pid), GRASS_KEYS) if pid in seg_data else None)


# Per-park visualization — original + segmented imagery side by side, then class breakdown bar.
for _, r in top_n.iterrows():
    pid = r.park_id
    if pid not in sat_imgs:
        continue
    imgs = sat_imgs[pid]
    orig_img  = _decode_b64(imgs.get('orig'))
    seg_img   = _decode_b64(imgs.get('seg'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if orig_img is not None: axes[0].imshow(orig_img)
    axes[0].set_title(f"#{int(r['rank'])} {pid} satellite ({r.latitude:.4f}, {r.longitude:.4f})")
    axes[0].axis('off')
    if seg_img is not None: axes[1].imshow(seg_img)
    axes[1].set_title('Segmented image')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()


# Stacked bar of every class for the top-N parks.
rows = [(pid, segs) for pid, segs in seg_data.items() if segs]
if rows:
    classes = sorted({c for _, s in rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.2, 0.7 * len(rows))))
    bottoms = [0.0] * len(rows)
    pids = [pid for pid, _ in rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of surrounding scene')
    ax.set_title('Surface composition at top-N parks')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


# Map of all top-N parks with the satellite footprint.
fmap_sat = folium.Map(
    location=[top_n['latitude'].mean(), top_n['longitude'].mean()],
    zoom_start=14, tiles='cartodbpositron',
)
for _, r in top_n.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{int(r['rank'])} {r.park_id} — {r['name']}<br/>"
               f"peak {r.peak_temp_c:.1f} °C<br/>"
               f"canopy {r.canopy_pct}% · impervious {r.impervious_pct}%"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_sat)
    folium.Circle(
        location=[r.latitude, r.longitude], radius=GRANULARITY_M / 2,
        color='red', fill=True, fill_opacity=0.15,
        popup=f'~{GRANULARITY_M}m tile',
    ).add_to(fmap_sat)
display(fmap_sat)

top_n[['rank','park_id','name','type','peak_temp_c','canopy_pct','impervious_pct','grass_pct']]


---
## Step 6b — Or: load cached satellite segmentation (for testing)

### What you are doing
Loading a single bundled satellite-segmentation file from `data/satellite/` and applying its percentages to all top-N parks for demonstration. On the live path each park is fetched independently.

### Why this matters
Use this path when iterating without burning API credits. The live cell saves per-park files under the same directory; pointing `SATELLITE_JSON` at any captured live file lets you replay one park's analysis without hitting the API again.

In [ ]:
import base64, io
from PIL import Image

IMPERV_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings',
               'rooftop', 'rooftops', 'sidewalk', 'earth', 'bare', 'ground'}
CANOPY_KEYS = {'tree', 'trees', 'vegetation', 'greenery', 'park'}
GRASS_KEYS  = {'grass'}


def _bucket(segments, keys):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)


def _first_b64(value):
    if isinstance(value, list): return value[0] if value else None
    return value


def _decode_b64(b64_str):
    if not b64_str: return None
    if isinstance(b64_str, list): b64_str = b64_str[0]
    if b64_str.startswith('data:'): b64_str = b64_str.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))


with open(SATELLITE_JSON, 'r', encoding='utf-8') as f:
    sat_doc = json.load(f)

seg_block = sat_doc.get('segmentation', {}) or {}
cached_segments = seg_block.get('segments', {}) or {}
cached_imgs = {'orig': _first_b64(sat_doc.get('orignal_image') or sat_doc.get('original_image')),
               'seg' : seg_block.get('image_content')}

top_n = parks.head(TOP_N_TO_ENRICH).copy()
seg_data = {pid: cached_segments for pid in top_n['park_id']}
sat_imgs = {pid: cached_imgs for pid in top_n['park_id']}
sat_source = {pid: 'cached sample' for pid in top_n['park_id']}

top_n['canopy_pct']     = _bucket(cached_segments, CANOPY_KEYS)
top_n['impervious_pct'] = _bucket(cached_segments, IMPERV_KEYS)
top_n['grass_pct']      = _bucket(cached_segments, GRASS_KEYS)
print(f'[cached] satellite sample applied to top {len(top_n)} parks for demo. '
      f'In live mode each park is fetched independently.')


# Decode and show the cached imagery.
orig_img = _decode_b64(cached_imgs.get('orig'))
seg_img  = _decode_b64(cached_imgs.get('seg'))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if orig_img is not None: axes[0].imshow(orig_img)
axes[0].set_title(f'Cached satellite sample · {SATELLITE_JSON.name}')
axes[0].axis('off')
if seg_img is not None: axes[1].imshow(seg_img)
axes[1].set_title('Segmented image')
axes[1].axis('off')
plt.tight_layout(); plt.show()


# Stacked bar of cached class breakdown (one bar — sample is shared across top-N).
if cached_segments:
    classes = sorted(cached_segments.keys())
    vals = [float(cached_segments[c]) for c in classes]
    fig, ax = plt.subplots(figsize=(8, 3.0))
    cum = 0.0
    for cls, v in zip(classes, vals):
        ax.barh(['cached sample'], [v], left=cum, label=cls)
        cum += v
    ax.set_xlabel('% of surrounding scene')
    ax.set_title('Surface composition at the cached sample tile')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

top_n[['rank','park_id','name','type','peak_temp_c','canopy_pct','impervious_pct','grass_pct']]


---
## Step 7a — Street-view ground truth (live, top-N)

### What you are doing
For each of the top-N hottest parks, calling `client.street_view_segmentation` for the front view and persisting the raw response under `data/street_view/`. We compute per-park sky / tree / building / road percentages.

### Why this matters
Satellite view shows *surroundings from above*. Street view shows *what a visitor at the park entrance actually sees*. That perspective is where you confirm whether a shade structure is feasible, whether there is room for trees, and whether the park already has self-shading from buildings.

In [ ]:
import base64, io
from PIL import Image

TREE_KEYS_SV     = {'tree', 'trees', 'vegetation', 'greenery', 'grass'}
BUILDING_KEYS_SV = {'building', 'buildings', 'wall'}
SKY_KEYS_SV      = {'sky'}
ROAD_KEYS_SV     = {'road', 'roads', 'pavement', 'sidewalk'}


def _share(segs, keys):
    total = 0.0
    for cls, pct in (segs or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)


def _decode(b64):
    if not b64: return None
    if isinstance(b64, list): b64 = b64[0]
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))


if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY. Run Step 7b instead.')

sv_data    = {}   # pid → segments dict
sv_imgs    = {}   # pid → {'orig','seg','date'}
sv_source  = {}

for _, r in top_n.iterrows():
    sv = submit_and_wait_quiet(
        client.street_view_segmentation,
        f"streetview #{int(r['rank'])} {r.park_id}",
        latitude=float(r.latitude), longitude=float(r.longitude),
        vertical_angle=5.0, horizontal_angle=0.0, back_view=False,
        skip_on_failure=True, timeout=240,
    )
    if sv is None:   # no street-view imagery / task failed — leave sv_* None for this park
        continue
    res = sv['result']

    # Persist the raw response.
    out_path = STREET_SEG_DIR / f'streetview_{r.park_id}_{STUDY_DATE}_live.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f'  saved → {out_path.relative_to(ROOT)}')

    front = res.get('front', {}) or {}
    sv_data[r.park_id] = front.get('segments', {}) or {}
    sv_imgs[r.park_id] = {
        'orig': front.get('original_image'),
        'seg' : front.get('segmented_image'),
        'date': front.get('image_date', 'n/a'),
        'legend': front.get('image_legend', {}) or {},
    }
    sv_source[r.park_id] = 'live'


# Compute per-park percentages and attach to top_n.
for col, keys in [
    ('sv_tree_pct',     TREE_KEYS_SV),
    ('sv_building_pct', BUILDING_KEYS_SV),
    ('sv_sky_pct',      SKY_KEYS_SV),
    ('sv_road_pct',     ROAD_KEYS_SV),
]:
    top_n[col] = top_n['park_id'].map(
        lambda pid: _share(sv_data.get(pid, {}), keys) if pid in sv_data else None)


# Per-park visualization — image side by side + breakdown bars summary.
for _, r in top_n.iterrows():
    pid = r.park_id
    if pid not in sv_imgs:
        continue
    imgs = sv_imgs[pid]
    orig_img = _decode(imgs.get('orig'))
    seg_img  = _decode(imgs.get('seg'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if orig_img is not None: axes[0].imshow(orig_img)
    axes[0].set_title(f"#{int(r['rank'])} {pid} street view ({r.latitude:.4f}, {r.longitude:.4f})")
    axes[0].axis('off')
    if seg_img is not None: axes[1].imshow(seg_img)
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()


# Stacked bar across top-N — same shape as Step 6.
rows = [(pid, segs) for pid, segs in sv_data.items() if segs]
if rows:
    classes = sorted({c for _, s in rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.2, 0.7 * len(rows))))
    bottoms = [0.0] * len(rows)
    pids = [pid for pid, _ in rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of street-view scene')
    ax.set_title('Street-view composition at top-N parks')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


# Map of all top-N parks.
fmap_street = folium.Map(
    location=[top_n['latitude'].mean(), top_n['longitude'].mean()],
    zoom_start=15, tiles='cartodbpositron',
)
for _, r in top_n.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{int(r['rank'])} {r.park_id} — {r['name']}<br/>"
               f"sky {r.sv_sky_pct}% · tree {r.sv_tree_pct}% · building {r.sv_building_pct}%"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_street)
    folium.Circle(
        location=[r.latitude, r.longitude], radius=GRANULARITY_M / 2,
        color='red', fill=True, fill_opacity=0.15,
        popup=f'~{GRANULARITY_M}m tile',
    ).add_to(fmap_street)
display(fmap_street)

top_n[['rank','park_id','name','type','sv_sky_pct','sv_tree_pct','sv_building_pct','sv_road_pct']]


---
## Step 7b — Or: load cached street-view segmentation (for testing)

### What you are doing
Loading a single bundled street-view file from `data/street_view/` and applying its percentages to all top-N parks for demonstration.

### Why this matters
Use this path when iterating without burning API credits. The live cell saves per-park files under the same directory; swap `STREETVIEW_JSON` to load any captured run.

In [ ]:
import base64, io
from PIL import Image

TREE_KEYS_SV     = {'tree', 'trees', 'vegetation', 'greenery', 'grass'}
BUILDING_KEYS_SV = {'building', 'buildings', 'wall'}
SKY_KEYS_SV      = {'sky'}
ROAD_KEYS_SV     = {'road', 'roads', 'pavement', 'sidewalk'}


def _share(segs, keys):
    total = 0.0
    for cls, pct in (segs or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)


def _decode(b64):
    if not b64: return None
    if isinstance(b64, list): b64 = b64[0]
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))


with open(STREETVIEW_JSON, 'r', encoding='utf-8') as f:
    sv_doc = json.load(f)

front = sv_doc.get('front') or {}
cached_segments = front.get('segments', {}) or {}
cached_imgs = {'orig': front.get('original_image'),
               'seg' : front.get('segmented_image'),
               'date': front.get('image_date', 'n/a')}

sv_data   = {pid: cached_segments for pid in top_n['park_id']}
sv_imgs   = {pid: cached_imgs    for pid in top_n['park_id']}
sv_source = {pid: 'cached sample' for pid in top_n['park_id']}

# Attach scalar columns for Step 9 / 10 / 11 compat.
sv_tree = _share(cached_segments, TREE_KEYS_SV)
sv_bldg = _share(cached_segments, BUILDING_KEYS_SV)
sv_sky  = _share(cached_segments, SKY_KEYS_SV)
sv_road = _share(cached_segments, ROAD_KEYS_SV)
top_n['sv_tree_pct']     = sv_tree
top_n['sv_building_pct'] = sv_bldg
top_n['sv_sky_pct']      = sv_sky
top_n['sv_road_pct']     = sv_road
print(f'[cached] street-view sample applied to top {len(top_n)} parks: '
      f'tree {sv_tree}% · building {sv_bldg}% · sky {sv_sky}% · road/sidewalk {sv_road}%')


# Decode and show the cached imagery.
orig_img = _decode(cached_imgs.get('orig'))
seg_img  = _decode(cached_imgs.get('seg'))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if orig_img is not None: axes[0].imshow(orig_img)
axes[0].set_title(f'Cached street view · {STREETVIEW_JSON.name}  (imagery {cached_imgs.get("date","n/a")})')
axes[0].axis('off')
if seg_img is not None: axes[1].imshow(seg_img)
axes[1].set_title('Segmentation')
axes[1].axis('off')
plt.tight_layout(); plt.show()


if cached_segments:
    classes = sorted(cached_segments.keys())
    vals = [float(cached_segments[c]) for c in classes]
    fig, ax = plt.subplots(figsize=(8, 3.0))
    cum = 0.0
    for cls, v in zip(classes, vals):
        ax.barh(['cached sample'], [v], left=cum, label=cls)
        cum += v
    ax.set_xlabel('% of street-view scene')
    ax.set_title('Street-view composition at the cached sample tile')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

top_n[['rank','park_id','name','type','sv_sky_pct','sv_tree_pct','sv_building_pct','sv_road_pct']]


---
## Step 8a — Diurnal driver profile (live, top-N)

### What you are doing
For each of the top-N hottest parks, calling `client.environmental_parameters` with `filter_type=3` (single day — covers the full 24 h; `start_time` is ignored) to get the full 24-hour diurnal series, persisting the raw response under `data/env_params/`, and computing peak heat-index / peak hour / hours-above-NOAA-Caution within the 10:00–18:00 use window.

### Why this matters
A park that is unbearable from 12:00–17:00 demands different intervention timing than one that peaks during evening shade. The hour-by-hour profile tells you *when* discomfort peaks and which fix (shade, splash pad, posted advisory) is the right one — and humidity at peak is what distinguishes "misting will work" from "misting will be ineffective".

In [ ]:
if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY. Run Step 8b instead.')

env_data    = {}
env_source  = {}

for _, r in top_n.iterrows():
    env = submit_and_wait_quiet(
        client.environmental_parameters,
        f"env-params #{int(r['rank'])} {r.park_id}",
        latitude=float(r.latitude), longitude=float(r.longitude),
        temperature=float(r.peak_temp_c),
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=3,                # single day — returns the full 24h diurnal series
    )
    res = env['result']

    # Persist the raw response.
    out_path = ENV_DIR / f'env_params_{r.park_id}_{STUDY_DATE}_live.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f'  saved → {out_path.relative_to(ROOT)}')

    locs = res.get('locations') or []
    if not locs:
        continue
    params = (locs[0] or {}).get('parameters', {}) or {}
    env_data[r.park_id] = {
        'heat_index_celsius'           : list(params.get('heat_index_celsius') or []),
        'apparent_temperature_celsius' : list(params.get('apparent_temperature_celsius') or []),
        'relative_humidity_percent'    : list(params.get('relative_humidity_percent') or []),
    }
    env_source[r.park_id] = 'live'


def _peak_metrics(series):
    """Return (peak_hi, peak_hour, hours_above_caution_in_use_window).
    Assumes hi is indexed 0..23 (full diurnal series)."""
    hi = series.get('heat_index_celsius') or []
    if not hi:
        return None, None, None
    if len(hi) >= 24:
        use_slice = hi[USE_HOUR_START:USE_HOUR_END + 1]
        peak = max(v for v in use_slice if v is not None)
        peak_h = USE_HOUR_START + use_slice.index(peak)
        above = sum(1 for v in use_slice if v is not None and v >= HI_CAUTION_C)
    else:
        peak = max(v for v in hi if v is not None)
        peak_h = hi.index(peak)
        above = sum(1 for v in hi if v is not None and v >= HI_CAUTION_C)
    return round(peak, 1), peak_h, above


for col in ('peak_heat_index_c', 'peak_hi_hour', 'hi_hours_above_caution'):
    top_n[col] = None
for pid, s in env_data.items():
    peak, peak_h, above = _peak_metrics(s)
    mask = top_n['park_id'] == pid
    top_n.loc[mask, 'peak_heat_index_c']      = peak
    top_n.loc[mask, 'peak_hi_hour']           = peak_h
    top_n.loc[mask, 'hi_hours_above_caution'] = above


# Plot heat-index curves for the top parks.
for _, r in top_n.iterrows():
    s = env_data.get(r.park_id)
    if not s or not s.get('heat_index_celsius'):
        continue
    hi   = s['heat_index_celsius']
    appt = s.get('apparent_temperature_celsius') or []
    rh   = s.get('relative_humidity_percent') or []
    full_day = len(hi) >= 24
    hours = list(range(len(hi))) if full_day else list(range(USE_HOUR_START, USE_HOUR_START + len(hi)))

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(hours, hi, label='Heat index (°C)', color='#d62728', marker='o', markersize=3)
    if appt:
        ax.plot(hours[:len(appt)], appt, label='Apparent temp (°C)', color='#ff7f0e',
                marker='o', markersize=3)
    ax.axhline(HI_CAUTION_C, color='#fdae61', linestyle='--', alpha=0.7,
               label=f'NOAA Caution ({HI_CAUTION_C:.0f} °C)')
    ax.axhline(HI_EXTREME_CAUTION_C, color='#e34a33', linestyle='--', alpha=0.7,
               label=f'NOAA Extreme Caution ({HI_EXTREME_CAUTION_C:.0f} °C)')
    ax.axvspan(USE_HOUR_START - 0.5, USE_HOUR_END + 0.5,
               color='#bbbbbb', alpha=0.15, label='use window 10–18')
    if rh:
        ax_rh = ax.twinx()
        ax_rh.plot(hours[:len(rh)], rh, color='#1f77b4', alpha=0.45, label='RH (%)')
        ax_rh.set_ylabel('RH (%)', color='#1f77b4')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('°C')
    ax.set_title(f"Diurnal drivers — #{int(r['rank'])} {r.park_id} {r['name']}  ·  {env_source.get(r.park_id,'?')}")
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


# Map of all top-N parks with their peak driver readout.
fmap_env = folium.Map(
    location=[top_n['latitude'].mean(), top_n['longitude'].mean()],
    zoom_start=14, tiles='cartodbpositron',
)
for _, r in top_n.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{int(r['rank'])} {r.park_id} — {r['name']}<br/>"
               f"peak HI {r.peak_heat_index_c} °C @ {int(r.peak_hi_hour) if pd.notna(r.peak_hi_hour) else '—'}:00<br/>"
               f"hours ≥ Caution: {int(r.hi_hours_above_caution) if pd.notna(r.hi_hours_above_caution) else '—'}"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_env)
display(fmap_env)

top_n[['rank','park_id','name','peak_temp_c','peak_heat_index_c','peak_hi_hour','hi_hours_above_caution']]


---
## Step 8b — Or: load cached env-parameters (for testing)

### What you are doing
Loading the bundled env-params file from `data/env_params/` and applying its hourly series to all top-N parks for demonstration. The cached file already covers the full diurnal range (00:00–23:00) so the heat-index curves and peak-hour detection work the same as on the live path.

### Why this matters
Use this path when iterating without burning API credits, or to replay a captured run by pointing `ENV_PARAMS_JSON` at any per-park file the live cell saved.

In [ ]:
with open(ENV_PARAMS_JSON, 'r', encoding='utf-8') as f:
    env_doc = json.load(f)

locs = env_doc.get('locations') or []
cached_env = None
if locs:
    params = (locs[0] or {}).get('parameters', {}) or {}
    cached_env = {
        'heat_index_celsius'           : list(params.get('heat_index_celsius') or []),
        'apparent_temperature_celsius' : list(params.get('apparent_temperature_celsius') or []),
        'relative_humidity_percent'    : list(params.get('relative_humidity_percent') or []),
    }

env_data   = {pid: cached_env for pid in top_n['park_id']} if cached_env else {}
env_source = {pid: 'cached sample' for pid in env_data}
print(f'[cached] env-params sample applied to top {len(top_n)} parks for demo.')


def _peak_metrics(series):
    hi = series.get('heat_index_celsius') or []
    if not hi:
        return None, None, None
    if len(hi) >= 24:
        use_slice = hi[USE_HOUR_START:USE_HOUR_END + 1]
        peak = max(v for v in use_slice if v is not None)
        peak_h = USE_HOUR_START + use_slice.index(peak)
        above = sum(1 for v in use_slice if v is not None and v >= HI_CAUTION_C)
    else:
        peak = max(v for v in hi if v is not None)
        peak_h = hi.index(peak)
        above = sum(1 for v in hi if v is not None and v >= HI_CAUTION_C)
    return round(peak, 1), peak_h, above


for col in ('peak_heat_index_c', 'peak_hi_hour', 'hi_hours_above_caution'):
    top_n[col] = None
for pid, s in env_data.items():
    peak, peak_h, above = _peak_metrics(s)
    mask = top_n['park_id'] == pid
    top_n.loc[mask, 'peak_heat_index_c']      = peak
    top_n.loc[mask, 'peak_hi_hour']           = peak_h
    top_n.loc[mask, 'hi_hours_above_caution'] = above


# Plot the cached heat-index curve once (same series for every top-N park).
if cached_env and cached_env['heat_index_celsius']:
    hi   = cached_env['heat_index_celsius']
    appt = cached_env['apparent_temperature_celsius']
    rh   = cached_env['relative_humidity_percent']
    full_day = len(hi) >= 24
    hours = list(range(len(hi))) if full_day else list(range(USE_HOUR_START, USE_HOUR_START + len(hi)))

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(hours, hi, label='Heat index (°C)', color='#d62728', marker='o', markersize=3)
    if appt:
        ax.plot(hours[:len(appt)], appt, label='Apparent temp (°C)', color='#ff7f0e',
                marker='o', markersize=3)
    ax.axhline(HI_CAUTION_C, color='#fdae61', linestyle='--', alpha=0.7,
               label=f'NOAA Caution ({HI_CAUTION_C:.0f} °C)')
    ax.axhline(HI_EXTREME_CAUTION_C, color='#e34a33', linestyle='--', alpha=0.7,
               label=f'NOAA Extreme Caution ({HI_EXTREME_CAUTION_C:.0f} °C)')
    ax.axvspan(USE_HOUR_START - 0.5, USE_HOUR_END + 0.5,
               color='#bbbbbb', alpha=0.15, label='use window 10–18')
    if rh:
        ax_rh = ax.twinx()
        ax_rh.plot(hours[:len(rh)], rh, color='#1f77b4', alpha=0.45, label='RH (%)')
        ax_rh.set_ylabel('RH (%)', color='#1f77b4')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('°C')
    ax.set_title('Diurnal drivers — cached sample (applied to all top-N for demo)')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

top_n[['rank','park_id','name','peak_temp_c','peak_heat_index_c','peak_hi_hour','hi_hours_above_caution']]


---
## Step 9 — Action brief (declarative, threshold-triggered)

### What you are doing
For each top-N park, we evaluate every measurement against its published threshold. **No invented index.** A recommendation appears in the brief only when a real measurement crosses a real threshold — and the brief cites the program that funds the fix.

### Why this matters (public health)
This step is the difference between *"this park scored 0.74"* and *"the canopy here is 14 % vs. an EPA / USDA target of 25 %, so file an i-Tree planting plan"*. The first is something the parks director cannot defend in a council meeting; the second is something they can hand to the grants officer.

Triggers and citations:

| If the API measures | Crossing threshold | Recommendation | Program |
|---|---|---|---|
| `canopy_pct` | < 25 (EPA / USDA target) | Tree-planting plan | USDA Forest Service i-Tree · EPA Heat Island Reduction |
| `impervious_pct` | > 60 (EPA Heat Island guidance) | Cool-pavement retrofit | EPA Heat Island Reduction |
| `peak_heat_index_c` | ≥ 32 (NOAA Extreme Caution) | Posted heat advisory + activity-window guidance | CDC BRACE |
| `sv_sky_pct` | > 60 with `sv_tree_pct` < 15 | Shade structure at entrance / play area | NRPA Shade-Equity guidance |
| RH > 60 % with peak temp > 30 °C | (compound) | Splash-pad / mister feasibility | CDC heat-resilience programs |

In [ ]:
def _triggers_for(row):
    """Return list of (label, evidence_string, program) tuples — a recommendation per threshold crossed."""
    triggers = []
    canopy = row.get('canopy_pct')
    imperv = row.get('impervious_pct')
    hi_pk  = row.get('peak_heat_index_c')
    sky    = row.get('sv_sky_pct')
    tree_sv = row.get('sv_tree_pct')
    s_env  = env_data.get(row['park_id']) or {}
    rh_series = s_env.get('relative_humidity_percent') or []
    rh_peak = max(rh_series) if rh_series else None

    if canopy is not None and not pd.isna(canopy) and canopy < CANOPY_TARGET_PCT:
        triggers.append((
            'Tree-planting plan',
            f'satellite canopy {canopy:.1f}% vs. EPA / USDA target {CANOPY_TARGET_PCT:.0f}%',
            PROG_USDA_ITREE,
        ))
    if imperv is not None and not pd.isna(imperv) and imperv > IMPERVIOUS_TRIGGER:
        triggers.append((
            'Cool-pavement retrofit',
            f'satellite impervious {imperv:.1f}% > EPA guidance {IMPERVIOUS_TRIGGER:.0f}%',
            PROG_EPA_HEATISLAND,
        ))
    if hi_pk is not None and not pd.isna(hi_pk) and hi_pk >= HI_EXTREME_CAUTION_C:
        triggers.append((
            'Posted heat advisory + activity-window guidance',
            f'env-params peak heat index {hi_pk:.1f}°C ≥ NOAA Extreme Caution {HI_EXTREME_CAUTION_C:.0f}°C',
            PROG_CDC_BRACE,
        ))
    if (sky is not None and not pd.isna(sky) and sky > SKY_TRIGGER_PCT and
        tree_sv is not None and not pd.isna(tree_sv) and tree_sv < 15.0):
        triggers.append((
            'Shade structure at entrance / play area',
            f'street-view sky {sky:.1f}% with tree {tree_sv:.1f}% (NRPA shade-equity criterion)',
            PROG_NRPA_SHADE,
        ))
    if (rh_peak is not None and rh_peak > RH_HIGH_PCT and
        row.get('peak_temp_c') is not None and row.get('peak_temp_c') > 30.0):
        triggers.append((
            'Splash-pad / mister feasibility study',
            f'peak RH {rh_peak:.0f}% with peak temp {row["peak_temp_c"]:.1f}°C',
            PROG_CDC_HEAT_RES,
        ))
    return triggers


def _peak_hour_label(peak_hour):
    """Format peak_hour for display — guards against None (live filter_type=3 path)."""
    if peak_hour is None or pd.isna(peak_hour):
        return '(daily peak)'
    return f'@ {int(peak_hour):02d}:00'


# Render an action brief per top-N park.
for _, r in top_n.iterrows():
    triggers = _triggers_for(r.to_dict() | {'park_id': r['park_id']})
    rank = int(r['rank'])
    header = (f'<div style="font:600 15px sans-serif;margin:14px 0 4px 0;color:#0d0d0d">'
              f'#{rank} · {r["park_id"]} — {r["name"]}'
              f'<span style="color:#555;font-weight:400"> · {r["type"]} · {r["acres"]:.1f} ac · '
              f'peak {r.peak_temp_c:.1f}°C {_peak_hour_label(r.peak_hour)}</span></div>')
    if not triggers:
        body = ('<div style="padding:10px 14px;border-left:4px solid #1a9850;background:#f7fbf7;'
                'border-radius:0 4px 4px 0;color:#1a1a1a">'
                '<b>No published threshold crossed.</b> Annual monitoring sufficient.'
                '</div>')
    else:
        items = ''.join(
            f'<div style="padding:10px 14px;border-left:4px solid #d73027;background:#fff5f5;'
            f'border-radius:0 4px 4px 0;margin-bottom:8px;color:#1a1a1a">'
            f'<div style="font:600 13px sans-serif;color:#0d0d0d">→ {label}</div>'
            f'<div style="margin:4px 0;color:#333">Trigger: {evidence}.</div>'
            f'<div style="font-size:11px;color:#666;">Program: {program}</div>'
            f'</div>'
            for label, evidence, program in triggers
        )
        body = items
    display(HTML('<div style="border:1px solid #ccc;border-radius:6px;padding:14px 16px;margin:10px 0;'
                 'font:13px/1.5 -apple-system,sans-serif;background:#ffffff">'
                 + header + body + '</div>'))


---
## Step 10 — Per-park audit CSV + ranked priority map (M3)

### What you are doing
Compose the final per-park audit row: every column is a direct API output, and the recommendation cell is the concatenated set of trigger-driven recommendations (or "Annual monitoring" if nothing crossed). Save to CSV. Render a ranked map with markers sized by peak temperature and popups carrying the recommendation list.

### Why this matters (public health)
The CSV is what the parks office adds to the next budget cycle. The map is what the public-health director shows the city council. Both end at the same content — the difference is medium, not data.

In [ ]:
# Merge the top-N enriched columns back into the full park frame.
merge_cols = ['park_id', 'canopy_pct', 'impervious_pct', 'grass_pct',
              'peak_heat_index_c', 'peak_hi_hour', 'hi_hours_above_caution',
              'sv_tree_pct', 'sv_building_pct', 'sv_sky_pct', 'sv_road_pct']
parks = parks.merge(top_n[merge_cols], on='park_id', how='left')

# For parks not in top-N, satellite/env-params columns are None — only heatmap-derived
# triggers will fire for them. That is the correct conservative behavior.
def _row_recommendation(row):
    triggers = _triggers_for(row)
    if not triggers:
        return 'Annual monitoring'
    return ' · '.join(f'{label} ({program.split(" · ")[0]})' for label, _, program in triggers)

parks['recommendation'] = parks.apply(
    lambda r: _row_recommendation(r.to_dict() | {'park_id': r['park_id']}), axis=1)

audit_cols = ['rank', 'park_id', 'name', 'type', 'acres',
              'peak_temp_c', 'peak_hour', 'daily_mean_temp_c', 'aoi_percentile',
              'hours_above_caution_in_use', 'use_window_peak_c',
              'canopy_pct', 'impervious_pct',
              'peak_heat_index_c', 'peak_hi_hour', 'hi_hours_above_caution',
              'sv_tree_pct', 'sv_sky_pct',
              'recommendation']
for c in audit_cols:
    if c not in parks.columns:
        parks[c] = None
audit = parks[audit_cols]
# CSV preserves the full audit (all parks); the on-screen map and table below show
# only the top-N that received the deeper satellite + street-view + env-params look.
audit.to_csv(OUTPUT_CSV, index=False)

n_recs = (parks['recommendation'] != 'Annual monitoring').sum()
print(f'Saved {len(audit)} parks to {OUTPUT_CSV}  (full audit covers every park)')
print(f'  {n_recs} of {len(parks)} parks have at least one threshold-triggered recommendation')
print(f'  Map and table below: top {TOP_N_TO_ENRICH} only')

# ── Priority tier per park, driven by the count of triggered recommendations ──
# 3+ triggers = highest priority, 2 = high, 1 = moderate, 0 = monitor.
# This connects the marker color directly to the declarative trigger logic in Step 9
# so the map legend names something the parks director can act on.
PRIORITY_TIERS = [
    ('Highest',  '#b30000', 14, 'three or more threshold-triggered recommendations'),
    ('High',     '#e34a33', 12, 'two threshold-triggered recommendations'),
    ('Moderate', '#fdae61', 10, 'one threshold-triggered recommendation'),
    ('Monitor',  '#1a9850',  6, 'no published threshold crossed'),
]
def _priority_for(rec_str):
    if rec_str == 'Annual monitoring':
        return PRIORITY_TIERS[3]                       # monitor
    n = rec_str.count(' · ') + 1
    if n >= 3: return PRIORITY_TIERS[0]                # highest
    if n == 2: return PRIORITY_TIERS[1]                # high
    return PRIORITY_TIERS[2]                            # moderate

top_audit = audit[audit['rank'] <= TOP_N_TO_ENRICH].copy()
top_audit['priority'] = top_audit['recommendation'].apply(lambda r: _priority_for(r)[0])

# ── M3 — final ranked map (top-N only) ────────────────────────
peak_lo = float(top_audit['peak_temp_c'].min())
peak_hi = float(top_audit['peak_temp_c'].max())
top_lats = top_audit.merge(parks[['park_id','latitude','longitude']], on='park_id')

m3_center = [top_lats['latitude'].mean(), top_lats['longitude'].mean()]
m3 = folium.Map(location=m3_center, zoom_start=12, tiles='cartodbpositron')
for _, p in top_lats.iterrows():
    priority_label, color, base_radius, _why = _priority_for(p['recommendation'])
    # Marker size carries peak-temperature (continuous signal) on top of the priority base size.
    if peak_hi > peak_lo:
        size_bump = (p['peak_temp_c'] - peak_lo) / (peak_hi - peak_lo) * 6
    else:
        size_bump = 0
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=base_radius + size_bump,
        color='black', weight=1.2,
        fill=True, fill_color=color, fill_opacity=0.92,
        tooltip=f"#{int(p['rank'])} {p['park_id']} — {priority_label} priority",
        popup=(f"<b>#{int(p['rank'])} {p['name']}</b><br/>"
               f"{p['type']} · {p['acres']:.1f} acres<br/>"
               f"<b>Priority: {priority_label}</b><br/>"
               f"peak: {p.peak_temp_c:.1f}°C "
               f"{'@ ' + format(int(p.peak_hour), '02d') + ':00' if pd.notna(p.peak_hour) else '(daily peak)'}<br/>"
               f"hours ≥ NOAA Caution (10–18): {int(p['hours_above_caution_in_use']) if pd.notna(p['hours_above_caution_in_use']) else '—'}<br/>"
               f"<b style='color:#a00;'>→ {p['recommendation']}</b>"),
    ).add_to(m3)

if len(top_lats):
    m3.fit_bounds([[top_lats['latitude'].min() - 0.005, top_lats['longitude'].min() - 0.005],
                   [top_lats['latitude'].max() + 0.005, top_lats['longitude'].max() + 0.005]])

# Legend — explains both color (priority tier) and marker size (peak temperature)
priority_rows = ''.join(
    f'<tr>'
    f'<td style="padding:2px 6px;"><span style="display:inline-block;width:14px;height:14px;'
    f'border-radius:50%;background:{color};border:1px solid #000;"></span></td>'
    f'<td style="padding:2px 8px;font-weight:600;">{label}</td>'
    f'<td style="padding:2px 6px;color:#555;">{why}</td>'
    f'</tr>'
    for label, color, _radius, why in PRIORITY_TIERS
)
m3.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;'
    'padding:10px 14px;border:1px solid #888;font:12px sans-serif;max-width:360px;">'
    '<b>Priority tier</b><br/>'
    '<span style="color:#666;">color = number of threshold-triggered recommendations from Step 9</span>'
    f'<table style="margin-top:6px;border-collapse:collapse;">{priority_rows}</table>'
    '<div style="margin-top:8px;color:#666;font-size:11px;">'
    '<b>Marker size</b> ∝ measured peak temperature (within the top-N range). '
    'Larger marker = hotter park within the same priority tier.'
    '</div>'
    f'<div style="margin-top:6px;color:#666;font-size:11px;">'
    f'Showing top {TOP_N_TO_ENRICH} of {len(parks)} parks (the ones that received the deep-dive in Steps 6–8). '
    f'Full audit in {OUTPUT_CSV.name}.'
    '</div>'
    '</div>'
))
display(m3)

# Display table — top-N only, with the priority column up front for at-a-glance reading.
display_cols = ['rank', 'park_id', 'name', 'type', 'priority',
                'peak_temp_c', 'peak_hour', 'hours_above_caution_in_use',
                'canopy_pct', 'impervious_pct', 'peak_heat_index_c',
                'sv_sky_pct', 'recommendation']
top_audit[display_cols]

---
## Step 11 — Package outputs (CSV + PDF + maps)

### What you are doing
Bundling everything the audit produced into a single hand-off folder under `outputs/parks_<STUDY_DATE>/`:

- `audit.csv` — the full per-park audit (already written by Step 10).
- `parks_report.pdf` — multi-page slide-deck-ready PDF with the heatmap summary, top-N satellite/street/env diagnoses, action briefs, and the priority-tier table.
- `maps/*.html` — every interactive folium map (M1 overview, M2 hot-exposure cluster, top-N satellite / street view / env params, M3 final priority) saved as standalone HTML, openable in any browser.

### Why this matters
Different stakeholders consume different formats. The parks-and-rec analytics team wants the CSV; the council wants a PDF for the agenda packet; the design team wants the interactive maps to zoom and click around. This step produces all three at once, named consistently, in one folder you can hand off.

In [ ]:
import io, base64, json
from PIL import Image as PILImage

try:
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle,
        PageBreak, CondPageBreak, KeepTogether,
    )
except ImportError:
    raise RuntimeError(
        "reportlab is required for the PDF export. "
        "Install with: pip install 'reportlab>=4.0.0' (or pip install -r requirements.txt)."
    )

OUT_DIR = OUTPUTS_ROOT
(OUT_DIR / 'maps').mkdir(parents=True, exist_ok=True)


# --- 1. Save folium maps as standalone HTML. --------------------------------
maps = [
    ('m1_park_network.html',     globals().get('m1')),
    ('m2_above_median.html',     globals().get('m2')),
    ('m3_priority_ranked.html',  globals().get('m3')),
    ('satellite_top_n.html',     globals().get('fmap_sat')),
    ('street_view_top_n.html',   globals().get('fmap_street')),
    ('env_params_top_n.html',    globals().get('fmap_env')),
]
for fname, m in maps:
    if m is None:
        continue
    m.save(str(OUT_DIR / 'maps' / fname))
    print(f'  ✓ maps/{fname}')


# --- 2. PDF report (ReportLab Platypus). ------------------------------------
PAGE_W, PAGE_H = letter
LEFT_MARGIN  = 0.6 * inch
RIGHT_MARGIN = 0.6 * inch
TOP_MARGIN   = 0.7 * inch
BOT_MARGIN   = 0.7 * inch
USABLE_W     = PAGE_W - LEFT_MARGIN - RIGHT_MARGIN

# Brand palette.
BRAND_BLUE   = colors.HexColor('#0E4A8A')
BRAND_INK    = colors.HexColor('#1f2933')
BRAND_MUTED  = colors.HexColor('#5a6b7b')
BRAND_YELLOW = colors.HexColor('#FFD24D')

LOGO_PATH        = ROOT / 'assets' / 'fortyguard_logo.png'
LOGO_FOOTER_PATH = ROOT / 'assets' / 'fortyguard_logo_footer.png'
COVER_BG_PATH    = ROOT / 'assets' / 'cover_bg.png'
LOGO_ASPECT        = 224 / 1208
LOGO_FOOTER_ASPECT = 68 / 364

REPORT_NAME = 'Public-Parks Heat-Resilience Audit'

_styles = getSampleStyleSheet()
S_H1    = ParagraphStyle('H1', parent=_styles['Heading1'],
                         fontName='Helvetica-Bold', fontSize=16, leading=20,
                         textColor=BRAND_BLUE, spaceBefore=4, spaceAfter=10)
S_H2    = ParagraphStyle('H2', parent=_styles['Heading2'],
                         fontName='Helvetica-Bold', fontSize=12, leading=15,
                         textColor=BRAND_INK, spaceBefore=4, spaceAfter=6)
S_BODY  = ParagraphStyle('Body', parent=_styles['BodyText'],
                         fontName='Helvetica', fontSize=10, leading=14,
                         textColor=BRAND_INK, spaceAfter=4)
S_BODY_W = ParagraphStyle('BodyW', parent=S_BODY, fontName='Helvetica', fontSize=8, leading=11)
S_CAP   = ParagraphStyle('Cap', parent=_styles['Italic'],
                         fontName='Helvetica-Oblique', fontSize=9, leading=12,
                         textColor=BRAND_MUTED, spaceAfter=8)
S_CONTACT = ParagraphStyle('Contact', parent=_styles['Normal'],
                           fontName='Helvetica', fontSize=9, leading=13,
                           textColor=BRAND_INK, spaceAfter=4)


def _h1(text):
    return Paragraph(str(text).upper().replace('&', '&amp;'), S_H1)


def _h2(text):
    return Paragraph(str(text).upper().replace('&', '&amp;'), S_H2)


def _fig_to_image(fig, max_width_inches=6.6, dpi=180):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    img = PILImage.open(buf)
    iw, ih = img.size
    aspect = ih / iw
    width  = max_width_inches * inch
    return RLImage(buf, width=width, height=width * aspect)


def _pil_to_image(pil_img, max_width_inches=6.6):
    if pil_img is None:
        return None
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    buf.seek(0)
    iw, ih = pil_img.size
    aspect = ih / iw
    width  = max_width_inches * inch
    return RLImage(buf, width=width, height=width * aspect)


def _decode_b64(b64_str):
    if not b64_str:
        return None
    if isinstance(b64_str, list):
        b64_str = b64_str[0]
    if b64_str.startswith('data:'):
        b64_str = b64_str.split(',', 1)[1]
    try:
        return PILImage.open(io.BytesIO(base64.b64decode(b64_str)))
    except Exception:
        return None


def _wrap(s, style=S_BODY):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return Paragraph('—', style)
    return Paragraph(str(s).replace('\n', '<br/>'), style)


TABLE_STYLE = TableStyle([
    ('BACKGROUND',    (0, 0),  (-1, 0),   BRAND_BLUE),
    ('TEXTCOLOR',     (0, 0),  (-1, 0),   colors.white),
    ('FONTNAME',      (0, 0),  (-1, 0),   'Helvetica-Bold'),
    ('FONTSIZE',      (0, 0),  (-1, 0),   9),
    ('ALIGN',         (0, 0),  (-1, 0),   'LEFT'),
    ('VALIGN',        (0, 0),  (-1, -1),  'MIDDLE'),
    ('LINEBELOW',     (0, 0),  (-1, 0),   0.6, BRAND_BLUE),
    ('LINEBELOW',     (0, -1), (-1, -1),  0.4, colors.HexColor('#cbd5e0')),
    ('FONTSIZE',      (0, 1),  (-1, -1),  8),
    ('LEFTPADDING',   (0, 0),  (-1, -1),  6),
    ('RIGHTPADDING',  (0, 0),  (-1, -1),  6),
    ('TOPPADDING',    (0, 0),  (-1, -1),  5),
    ('BOTTOMPADDING', (0, 0),  (-1, -1),  5),
    ('ROWBACKGROUNDS',(0, 1),  (-1, -1),  [colors.white, colors.HexColor('#f4f7fb')]),
])


def _draw_cover(canvas, doc):
    """Page 1: full-bleed bg.png, yellow accents, white wordmark at lower-left."""
    canvas.saveState()

    if COVER_BG_PATH.exists():
        canvas.drawImage(str(COVER_BG_PATH), 0, 0,
                         width=PAGE_W, height=PAGE_H,
                         preserveAspectRatio=False, mask='auto')
    else:
        canvas.setFillColor(BRAND_BLUE)
        canvas.rect(0, 0, PAGE_W, PAGE_H, stroke=0, fill=1)

    # Yellow pill — study date.
    pill_h = 0.34 * inch
    pill_w = 1.45 * inch
    pill_x = LEFT_MARGIN
    pill_y = PAGE_H - 1.05 * inch
    canvas.setFillColor(BRAND_YELLOW)
    canvas.roundRect(pill_x, pill_y, pill_w, pill_h, pill_h / 2,
                     stroke=0, fill=1)
    canvas.setFillColor(BRAND_INK)
    canvas.setFont('Helvetica-Bold', 11)
    canvas.drawCentredString(pill_x + pill_w / 2,
                             pill_y + pill_h / 2 - 0.04 * inch,
                             STUDY_DATE)

    # Title — three short uppercase lines.
    canvas.setFillColor(colors.white)
    canvas.setFont('Helvetica-Bold', 38)
    title_top_y = PAGE_H - 1.8 * inch
    line_h      = 0.55 * inch
    canvas.drawString(LEFT_MARGIN, title_top_y - 0 * line_h, 'PUBLIC PARKS')
    canvas.drawString(LEFT_MARGIN, title_top_y - 1 * line_h, 'HEAT-RESILIENCE')
    canvas.drawString(LEFT_MARGIN, title_top_y - 2 * line_h, 'AUDIT')

    def _info_line(y, label, value):
        canvas.setFont('Helvetica-Bold', 11)
        canvas.setFillColor(BRAND_YELLOW)
        canvas.drawString(LEFT_MARGIN, y, label)
        label_w = canvas.stringWidth(label, 'Helvetica-Bold', 11)
        canvas.setFont('Helvetica', 11)
        canvas.setFillColor(colors.white)
        canvas.drawString(LEFT_MARGIN + label_w + 0.06 * inch, y, value)

    info_y = title_top_y - 2 * line_h - 0.55 * inch
    _info_line(info_y, 'AOI:',
               f' {len(tiles):,} tiles at {GRANULARITY_M} m  ·  '
               f'{len(parks)} parks  ·  top {TOP_N_TO_ENRICH} diagnosed in detail')

    aoi_peaks = [t[2] for t in tiles]
    info2_y = info_y - 0.28 * inch
    if aoi_peaks:
        aoi_min  = min(aoi_peaks)
        aoi_max  = max(aoi_peaks)
        aoi_mean = sum(aoi_peaks) / len(aoi_peaks)
        _info_line(info2_y, 'AOI peak temperature range:',
                   f' {aoi_min:.1f} – {aoi_max:.1f} °C  (mean {aoi_mean:.1f} °C)')

    upper_dots_y = (info2_y if aoi_peaks else info_y) - 0.40 * inch
    for i in range(3):
        canvas.setFillColor(BRAND_YELLOW)
        canvas.circle(LEFT_MARGIN + 0.10 * inch + i * 0.32 * inch, upper_dots_y,
                      0.10 * inch, stroke=0, fill=1)

    logo_w = 3.6 * inch
    logo_h = logo_w * LOGO_ASPECT
    logo_y = 1.2 * inch
    if LOGO_PATH.exists():
        canvas.drawImage(str(LOGO_PATH), LEFT_MARGIN, logo_y,
                         width=logo_w, height=logo_h, mask='auto')

    lower_dots_y = logo_y + logo_h + 0.28 * inch
    for i in range(3):
        canvas.setFillColor(BRAND_YELLOW)
        canvas.circle(LEFT_MARGIN + 0.10 * inch + i * 0.32 * inch, lower_dots_y,
                      0.10 * inch, stroke=0, fill=1)

    canvas.restoreState()


def _draw_body(canvas, doc):
    canvas.saveState()
    canvas.setStrokeColor(colors.HexColor('#cbd5e0'))
    canvas.setLineWidth(0.4)
    canvas.line(LEFT_MARGIN, 0.6 * inch, PAGE_W - RIGHT_MARGIN, 0.6 * inch)

    if LOGO_FOOTER_PATH.exists():
        foot_w = 0.85 * inch
        foot_h = foot_w * LOGO_FOOTER_ASPECT
        canvas.drawImage(str(LOGO_FOOTER_PATH),
                         LEFT_MARGIN, 0.32 * inch,
                         width=foot_w, height=foot_h, mask='auto')

    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(BRAND_MUTED)
    canvas.drawString(LEFT_MARGIN + 1.0 * inch, 0.4 * inch,
                      f'{REPORT_NAME}  ·  {STUDY_DATE}')
    canvas.drawRightString(PAGE_W - RIGHT_MARGIN, 0.4 * inch, f'Page {doc.page}')
    canvas.restoreState()


# ── Matplotlib figure builders. ────────────────────────────────────────────
def _build_heatmap_distribution_fig():
    temps = [t[2] for t in tiles]
    if not temps:
        return None
    lo, hi, mean = float(min(temps)), float(max(temps)), float(sum(temps) / len(temps))

    fig = plt.figure(figsize=(11, 4.5), constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    from matplotlib.patches import FancyBboxPatch
    ax0.add_patch(FancyBboxPatch(
        (-0.02, -0.02), 1.04, 1.04,
        transform=ax0.transAxes,
        boxstyle='round,pad=0.02,rounding_size=0.04',
        facecolor='white', edgecolor='#d8dee6', linewidth=0.8,
        clip_on=False, zorder=0,
    ))
    ax0.text(0.05, 0.92, f'Heatmap · {STUDY_DATE}', transform=ax0.transAxes,
             fontsize=12, fontweight='bold', va='top', zorder=2)
    ax0.text(0.05, 0.82, f'{len(temps):,} tiles', transform=ax0.transAxes,
             fontsize=10, color='#666', va='top', zorder=2)
    rows = [('min', lo), ('mean', mean), ('max', hi)]
    y = 0.62
    for label, val in rows:
        ax0.text(0.05, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace', zorder=2)
        ax0.add_patch(plt.Rectangle((0.27, y - 0.06), 0.10, 0.12,
                                    transform=ax0.transAxes,
                                    facecolor=temp_color(val, lo, hi),
                                    edgecolor='#333', linewidth=0.6, zorder=2))
        ax0.text(0.42, y, f'{val:.2f} °C', transform=ax0.transAxes,
                 fontsize=12, fontweight='bold', va='center', family='monospace', zorder=2)
        y -= 0.18

    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, e_lo, e_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((e_lo + e_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile peak temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Daily peak temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for sp in ('top', 'right'):
        ax1.spines[sp].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP, extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([]); ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)
    return fig


def _build_seg_breakdown_fig(segments, title, color='#3b8686'):
    if not segments:
        return None
    items   = sorted(segments.items(), key=lambda x: float(x[1]), reverse=True)
    classes = [k for k, _ in items]
    pcts    = [float(v) for _, v in items]
    fig, ax = plt.subplots(figsize=(8.5, max(2.5, 0.4 + 0.35 * len(classes))))
    ax.barh(classes, pcts, color=color, edgecolor='#333', linewidth=0.6)
    for i, p in enumerate(pcts):
        ax.text(p + max(pcts) * 0.015, i, f'{p:.1f}%',
                va='center', fontsize=9, fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlabel('Coverage (%)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


def _build_top_n_surface_comp_fig(seg_dict, title):
    rows = [(pid, segs) for pid, segs in (seg_dict or {}).items() if segs]
    if not rows:
        return None
    classes = sorted({c for _, s in rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(9, max(2.5, 0.7 * len(rows))))
    bottoms = [0.0] * len(rows)
    pids = [pid for pid, _ in rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of scene')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout()
    return fig


def _build_diurnal_fig(s, title):
    """Heat-index / apparent / RH curves with NOAA caution + use-window markers."""
    if not s or not s.get('heat_index_celsius'):
        return None
    hi_  = s['heat_index_celsius']
    appt = s.get('apparent_temperature_celsius') or []
    rh   = s.get('relative_humidity_percent') or []
    full_day = len(hi_) >= 24
    hours = list(range(len(hi_))) if full_day else list(range(USE_HOUR_START, USE_HOUR_START + len(hi_)))

    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(hours, hi_, label='Heat index (°C)', color='#d62728', marker='o', markersize=3)
    if appt:
        ax.plot(hours[:len(appt)], appt, label='Apparent temp (°C)', color='#ff7f0e',
                marker='o', markersize=3)
    ax.axhline(HI_CAUTION_C, color='#fdae61', linestyle='--', alpha=0.7,
               label=f'NOAA Caution ({HI_CAUTION_C:.0f} °C)')
    ax.axhline(HI_EXTREME_CAUTION_C, color='#e34a33', linestyle='--', alpha=0.7,
               label=f'NOAA Extreme Caution ({HI_EXTREME_CAUTION_C:.0f} °C)')
    ax.axvspan(USE_HOUR_START - 0.5, USE_HOUR_END + 0.5,
               color='#bbbbbb', alpha=0.15, label='use window 10–18')
    if rh:
        ax_rh = ax.twinx()
        ax_rh.plot(hours[:len(rh)], rh, color='#1f77b4', alpha=0.45, label='RH (%)')
        ax_rh.set_ylabel('RH (%)', color='#1f77b4')
    ax.set_xlabel('Hour of day'); ax.set_ylabel('°C')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig


# ── Build flowables. ───────────────────────────────────────────────────────
flowables = []

# Cover drawn on canvas.
flowables.append(PageBreak())

# Heatmap distribution
flowables.append(_h1('AOI peak-temperature distribution'))
flowables.append(Paragraph(
    'Per-tile daily peak across the AOI. The dashed line marks the AOI mean; '
    'the histogram is colored on the same spectral ramp as every map and chart in this report.',
    S_BODY))
hm_fig = _build_heatmap_distribution_fig()
if hm_fig is not None:
    flowables.append(Spacer(1, 0.1 * inch))
    flowables.append(_fig_to_image(hm_fig, max_width_inches=USABLE_W / inch))
flowables.append(Spacer(1, 0.35 * inch))
flowables.append(CondPageBreak(3.5 * inch))

# Top-N at a glance
flowables.append(_h1(f'Top {TOP_N_TO_ENRICH} hottest parks — at a glance'))
flowables.append(Paragraph(
    'Daily peak temperature, surface diagnosis (canopy / impervious %), '
    'street-view sky %, and peak heat index for each of the top-N parks.',
    S_BODY))

top_cols = ['Rank', 'ID', 'Name', 'Type', 'Acres', 'Peak °C',
            'Canopy %', 'Imperv %', 'Sky %', 'HI °C', 'Hrs ≥ Caut.']
top_rows = []
for _, r in top_n.iterrows():
    def _f(v, fmt='{:.1f}'):
        return fmt.format(v) if pd.notna(v) else '—'
    top_rows.append([
        f'#{int(r["rank"])}',
        r.park_id,
        _wrap(r['name'], S_BODY),
        r['type'],
        _f(r.get('acres')),
        _f(r.peak_temp_c),
        _f(r.get('canopy_pct'), '{:.0f}'),
        _f(r.get('impervious_pct'), '{:.0f}'),
        _f(r.get('sv_sky_pct'), '{:.0f}'),
        _f(r.get('peak_heat_index_c')),
        _f(r.get('hi_hours_above_caution'), '{:.0f}'),
    ])
top_table = Table([top_cols] + top_rows, repeatRows=1, hAlign='LEFT',
                  colWidths=[0.4*inch, 0.5*inch, 1.45*inch, 0.7*inch, 0.5*inch,
                             0.55*inch, 0.55*inch, 0.55*inch, 0.5*inch, 0.5*inch, 0.65*inch])
top_table.setStyle(TABLE_STYLE)
flowables.append(Spacer(1, 0.15 * inch))
flowables.append(top_table)
flowables.append(Spacer(1, 0.4 * inch))
flowables.append(CondPageBreak(4 * inch))

# Surface composition (C2 / C2b)
flowables.append(_h1('Surface composition across top-N (C2 / C2b)'))
flowables.append(Paragraph(
    'Stacked bars show the per-park class breakdown for satellite (overhead) '
    'and street view (front-of-park). Together they pin down whether the heat '
    'driver is impervious surface, low canopy, or a sky-exposed approach.',
    S_BODY))

c2_fig = _build_top_n_surface_comp_fig(seg_data if 'seg_data' in dir() else None,
                                        'C2 — Satellite surface composition (top-N)')
if c2_fig is not None:
    flowables.append(Spacer(1, 0.1 * inch))
    flowables.append(_fig_to_image(c2_fig, max_width_inches=USABLE_W / inch))

c2b_fig = None
if 'sv_data' in dir() and sv_data:
    sv_segs = {pid: (d.get('segs') if isinstance(d, dict) else d) or {}
               for pid, d in sv_data.items()}
    c2b_fig = _build_top_n_surface_comp_fig(sv_segs,
                                             'C2b — Street-view scene composition (top-N)')
if c2b_fig is not None:
    flowables.append(Spacer(1, 0.2 * inch))
    flowables.append(_fig_to_image(c2b_fig, max_width_inches=USABLE_W / inch))
flowables.append(Spacer(1, 0.4 * inch))

# Per-park deep dive — each park starts a fresh page; sub-sections flow continuously.
for _, r in top_n.iterrows():
    pid = r.park_id
    flowables.append(PageBreak())
    flowables.append(_h1(f"#{int(r['rank'])} · {pid} — {r['name']}"))

    metric_lines = [
        f"<b>Type:</b> {r['type']} &nbsp;·&nbsp; "
        f"<b>Acres:</b> {r.get('acres', '—')} &nbsp;·&nbsp; "
        f"<b>Peak temp:</b> {r.peak_temp_c:.1f} °C"
        + (f" @ {int(r.peak_hour):02d}:00" if pd.notna(r.peak_hour) else " (daily peak)")
        + (f" &nbsp;·&nbsp; <b>AOI percentile:</b> {r.aoi_percentile:.0f}"
           if pd.notna(r.get('aoi_percentile')) else "")
    ]
    if pd.notna(r.get('canopy_pct')) or pd.notna(r.get('impervious_pct')):
        metric_lines.append(
            f"<b>Surface:</b> canopy "
            f"{r.canopy_pct if pd.notna(r.canopy_pct) else '—'}% &nbsp;·&nbsp; "
            f"impervious {r.impervious_pct if pd.notna(r.impervious_pct) else '—'}% &nbsp;·&nbsp; "
            f"grass {r.grass_pct if pd.notna(r.get('grass_pct')) else '—'}%"
        )
    if pd.notna(r.get('peak_heat_index_c')):
        metric_lines.append(
            f"<b>Peak heat index:</b> {r.peak_heat_index_c:.1f} °C "
            f"&nbsp;·&nbsp; <b>Hours ≥ NOAA Caution:</b> "
            f"{int(r.hi_hours_above_caution) if pd.notna(r.hi_hours_above_caution) else '—'}"
        )
    for line in metric_lines:
        flowables.append(Paragraph(line, S_BODY))
    flowables.append(Spacer(1, 0.1 * inch))

    # Satellite — side-by-side pair.
    if pid in (sat_imgs or {}):
        imgs = sat_imgs[pid]
        orig_pil = _decode_b64(imgs.get('orig'))
        seg_pil  = _decode_b64(imgs.get('seg'))
        if orig_pil is not None or seg_pil is not None:
            flowables.append(_h2('Satellite — original & segmentation'))
            cells = []
            if orig_pil is not None:
                cells.append([_pil_to_image(orig_pil, max_width_inches=3.1),
                              Paragraph('original satellite tile', S_CAP)])
            else:
                cells.append([Paragraph('(original missing)', S_CAP)])
            if seg_pil is not None:
                cells.append([_pil_to_image(seg_pil, max_width_inches=3.1),
                              Paragraph('segmentation overlay', S_CAP)])
            else:
                cells.append([Paragraph('(segmentation missing)', S_CAP)])
            sat_row = Table([[c[0] for c in cells], [c[1] for c in cells]],
                            colWidths=[3.3 * inch, 3.3 * inch])
            sat_row.setStyle(TableStyle([
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('ALIGN',  (0, 0), (-1, -1), 'CENTER'),
                ('LEFTPADDING', (0, 0), (-1, -1), 0),
                ('RIGHTPADDING',(0, 0), (-1, -1), 0),
                ('TOPPADDING',  (0, 0), (-1, -1), 0),
                ('BOTTOMPADDING',(0, 0), (-1, -1), 0),
            ]))
            flowables.append(sat_row)
            flowables.append(Spacer(1, 0.1 * inch))
    if pid in (seg_data or {}) and seg_data[pid]:
        cb_fig = _build_seg_breakdown_fig(seg_data[pid],
                                           f'Satellite class breakdown — {pid}',
                                           color='#3b8686')
        if cb_fig is not None:
            flowables.append(_fig_to_image(cb_fig, max_width_inches=USABLE_W / inch))
    flowables.append(Spacer(1, 0.3 * inch))
    flowables.append(CondPageBreak(4 * inch))

    # Street view — side-by-side pair.
    flowables.append(_h1(f"#{int(r['rank'])} · {pid} — {r['name']} (street view)"))
    if pid in (sv_data or {}):
        d = sv_data[pid]
        if isinstance(d, dict):
            orig_pil = _decode_b64(d.get('orig'))
            seg_pil  = _decode_b64(d.get('seg'))
            segs     = d.get('segs', {}) or {}
            img_date = d.get('image_date', 'n/a')
        else:
            orig_pil = seg_pil = None
            segs = d if isinstance(d, dict) else {}
            img_date = 'n/a'
        flowables.append(Paragraph(f"Imagery date: {img_date}", S_BODY))
        if orig_pil is not None or seg_pil is not None:
            cells = []
            if orig_pil is not None:
                cells.append([_pil_to_image(orig_pil, max_width_inches=3.1),
                              Paragraph('original street-view (front)', S_CAP)])
            else:
                cells.append([Paragraph('(original missing)', S_CAP)])
            if seg_pil is not None:
                cells.append([_pil_to_image(seg_pil, max_width_inches=3.1),
                              Paragraph('pixel-wise segmentation', S_CAP)])
            else:
                cells.append([Paragraph('(segmentation missing)', S_CAP)])
            sv_row = Table([[c[0] for c in cells], [c[1] for c in cells]],
                           colWidths=[3.3 * inch, 3.3 * inch])
            sv_row.setStyle(TableStyle([
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('ALIGN',  (0, 0), (-1, -1), 'CENTER'),
                ('LEFTPADDING',  (0, 0), (-1, -1), 0),
                ('RIGHTPADDING', (0, 0), (-1, -1), 0),
                ('TOPPADDING',   (0, 0), (-1, -1), 0),
                ('BOTTOMPADDING',(0, 0), (-1, -1), 0),
            ]))
            flowables.append(sv_row)
            flowables.append(Spacer(1, 0.1 * inch))
        if segs:
            sv_cb_fig = _build_seg_breakdown_fig(segs,
                                                  f'Street-view class breakdown — {pid}',
                                                  color='#7a5195')
            if sv_cb_fig is not None:
                flowables.append(_fig_to_image(sv_cb_fig, max_width_inches=USABLE_W / inch))
    flowables.append(Spacer(1, 0.3 * inch))
    flowables.append(CondPageBreak(3.5 * inch))

    # Env-params diurnal
    flowables.append(_h1(f"#{int(r['rank'])} · {pid} — {r['name']} (env-params)"))
    s_env = (env_data or {}).get(pid)
    if s_env and s_env.get('heat_index_celsius'):
        flowables.append(Paragraph(
            f"Diurnal heat-index, apparent temperature, RH curves with the NOAA Caution / "
            f"Extreme-Caution thresholds and the {USE_HOUR_START:02d}:00–{USE_HOUR_END:02d}:00 use window marked.",
            S_BODY))
        ev_fig = _build_diurnal_fig(s_env,
                                     f"Diurnal drivers — #{int(r['rank'])} {pid}")
        if ev_fig is not None:
            flowables.append(_fig_to_image(ev_fig, max_width_inches=USABLE_W / inch))
    else:
        flowables.append(Paragraph('No env-params data available for this park.', S_BODY))

    # Triggers / recommendations
    triggers = _triggers_for(r.to_dict() | {'park_id': pid})
    flowables.append(Spacer(1, 0.15 * inch))
    flowables.append(_h2('Recommended actions'))
    if not triggers:
        flowables.append(Paragraph(
            '<font color="#1a9850"><b>OK</b></font> No published threshold crossed. '
            'Annual monitoring sufficient.',
            S_BODY))
    else:
        for label, evidence, program in triggers:
            flowables.append(Paragraph(f"<b>→ {label}</b>", S_BODY))
            flowables.append(Paragraph(f"Trigger: {evidence}.", S_BODY))
            flowables.append(Paragraph(f"<i>Program: {program}</i>", S_BODY))
            flowables.append(Spacer(1, 0.05 * inch))

# Consolidated action briefs across full audit
flowables.append(PageBreak())
flowables.append(_h1('Action briefs — consolidated'))
flowables.append(Paragraph(
    f'One row per park. Recommendations are derived from published thresholds (NOAA, EPA, USDA, NRPA, CDC). '
    f'Parks with no published threshold crossed receive annual-monitoring guidance.',
    S_BODY))

brief_cols = ['Rank', 'ID', 'Name', 'Type', 'Peak °C', 'Recommendation']
brief_rows = []
for _, p in audit.iterrows():
    def _f(v, fmt='{:.1f}'):
        return fmt.format(v) if pd.notna(v) else '—'
    brief_rows.append([
        int(p['rank']) if pd.notna(p['rank']) else '—',
        p.park_id,
        _wrap(p['name'], S_BODY_W),
        p.get('type', '—'),
        _f(p.peak_temp_c),
        _wrap(p.get('recommendation', '—'), S_BODY_W),
    ])
brief_table = Table([brief_cols] + brief_rows, repeatRows=1, hAlign='LEFT',
                    colWidths=[0.4*inch, 0.45*inch, 1.4*inch, 0.85*inch,
                               0.6*inch, 3.6*inch])
brief_table.setStyle(TABLE_STYLE)
flowables.append(Spacer(1, 0.15 * inch))
flowables.append(brief_table)

# Closing contact block — yellow rule, contact heading + text, centered blue wordmark.
yellow_rule = Table([['']], colWidths=[USABLE_W], rowHeights=[0.06 * inch])
yellow_rule.setStyle(TableStyle([
    ('BACKGROUND',    (0, 0), (-1, -1), BRAND_YELLOW),
    ('LEFTPADDING',   (0, 0), (-1, -1), 0),
    ('RIGHTPADDING',  (0, 0), (-1, -1), 0),
    ('TOPPADDING',    (0, 0), (-1, -1), 0),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 0),
]))

closing_block = [
    Spacer(1, 0.45 * inch),
    yellow_rule,
    Spacer(1, 0.25 * inch),
    _h1('FortyGuard contact details'),
    Paragraph(
        'FortyGuard Tech Limited, Al Khatem Tower, 14th Floor, 112, WeWork Hub71, '
        'Abu Dhabi Global Market Square, Al Maryah Island, Abu Dhabi, UAE. '
        'PoBox: 3317, Tel: +97126662799, Email: info@fortyguard.com',
        S_CONTACT),
    Spacer(1, 0.35 * inch),
]

if LOGO_FOOTER_PATH.exists():
    closing_logo_w = 1.8 * inch
    closing_logo_h = closing_logo_w * LOGO_FOOTER_ASPECT
    closing_logo   = RLImage(str(LOGO_FOOTER_PATH),
                             width=closing_logo_w, height=closing_logo_h)
    closing_logo.hAlign = 'CENTER'
    closing_block.append(closing_logo)

flowables.append(KeepTogether(closing_block))


# ── Render the PDF. ───────────────────────────────────────────────────────
pdf_path = OUT_DIR / 'parks_report.pdf'
doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=letter,
    leftMargin=LEFT_MARGIN, rightMargin=RIGHT_MARGIN,
    topMargin=TOP_MARGIN,   bottomMargin=BOT_MARGIN,
    title=f'{REPORT_NAME} · {STUDY_DATE}',
    author='FortyGuard',
)
doc.build(flowables, onFirstPage=_draw_cover, onLaterPages=_draw_body)


# Save the audit CSV (Step 10 also saves it; this guarantees the file is in the bundle).
audit_csv_path = OUT_DIR / 'audit.csv'
if 'audit' in dir():
    audit.to_csv(audit_csv_path, index=False)
    print(f'  ✓ {audit_csv_path.relative_to(ROOT)}')

print(f'  ✓ {pdf_path.relative_to(ROOT)}')
print(f'\nFull bundle: {OUT_DIR.relative_to(ROOT)}')
print(f'  - audit.csv')
print(f'  - parks_report.pdf')
print(f'  - maps/*.html  (open any in a browser)')


---
## Wrap-up

Starting from a parks point CSV you now have:

| Artifact | Audience |
|----------|----------|
| Per-park measurement table (every column is a direct API output) | Parks-and-rec analytics |
| **M1** park-network overview map | Briefing slide 1 |
| **M2** above-median hot exposures map | Briefing slide 2 |
| **M3** final ranked priority map with recommendations in the popup | Council meeting |
| Surface-composition stacked bar (top-N) | Grant-writing pack |
| Diurnal heat-index curves (top-N) | Posted heat-advisory documentation |
| Per-park action brief HTML cards | Parks director — go/no-go for each grant application |
| **Bundled hand-off folder (CSV + PDF report + interactive maps)** | **Slide decks, council packet, ops** |

### Where everything lives on disk

```
data/
  heatmaps/       ← raw heatmap GeoJSON outputs (live + cached)
  satellite/      ← raw satellite-segmentation JSON per park (live + cached)
  street_view/    ← raw street-view JSON per park (live + cached)
  env_params/     ← raw env-params JSON per park (live + cached)
outputs/
  parks_<STUDY_DATE>/
    audit.csv
    parks_report.pdf      ← multi-page PDF for slide decks
    maps/*.html           ← interactive folium maps (M1, M2, M3 + per-step diagnostic maps)
```

Re-running against any captured live response is a one-liner: change the filename in the matching cache cell (Step 2b / 6b / 7b / 8b) to point at any file in the corresponding `data/<type>/` subfolder.

**No invented index.** Every column is a direct API output. **No dollar value.** Every recommendation is keyed to a published national program (NOAA Heat Index, EPA Heat Island Reduction, USDA Forest Service i-Tree, CDC BRACE, NRPA Shade-Equity, CDC heat-resilience).

**Apply this pattern to adjacent use cases**: outdoor public-school zones (recess + PE + drop-off shade), public-library outdoor courtyards, public-housing common areas, faith-based / community-center outdoor gathering spaces, public swimming-pool approaches and queues. The workflow — *public point list × diurnal heatmap × surface diagnosis × ground-truth × env-params → declarative trigger-recommendation table* — transfers directly to anywhere the public stands outside in the heat.